# Panel Lémanique — Walking behaviour analysis

In [ ]:
import geopandas as gpd
import pandas as pd
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import numpy as np
import matplotlib.patches as mpatches
from adjustText import adjust_text

operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

input_file_path_PL = '/Volumes/T7_lin_win/PANEL_LEMANIQUE/'
output_file_path_PL = '/Volumes/T7_lin_win/PANEL_LEMANIQUE/'
input_file_path = '../../Data/input/'
output_file_path = '../../Data/output/'

#Read the files
agglo_commune = gpd.read_file(f'{input_file_path_PL}WAVE_RYTHM/INPUT/AGGLO_COMMUNES-SHP/AGGLO_COMMUNES.shp')
agglo_commune = agglo_commune.to_crs(operation_crs)

zones_girec = gpd.read_file(f'{input_file_path_PL}GPS_tracking_data_PL/OUTPUT/zones_girec.gpkg') #zones GIREC provenant directement du code 3_2 comme ça fichier directement enrichis des données ajoutées dans ce code
zones_girec = zones_girec.to_crs(operation_crs)

commune_GE = gpd.read_file(f'{input_file_path_PL}WAVE_RYTHM/INPUT/CAD_COMMUNE-SHP/CAD_COMMUNE.shp')
commune_GE = commune_GE.to_crs(operation_crs)

PL_respondant_wave1 = gpd.read_file(f'{input_file_path_PL}WAVE1_MOBILITY/OUTPUT/respondants_all_wave1.gpkg')
PL_respondant_wave1 = PL_respondant_wave1.to_crs(operation_crs)

step3_index = gpd.read_file(f'{output_file_path}GE/step-3/step3_index.gpkg')
step3_index = step3_index.to_crs(operation_crs)

legs_GE_walk_users = gpd.read_file(f'{output_file_path_PL}GPS_tracking_data_PL/OUTPUT/legs_GE_walk_user.gpkg') #from notebook gps_data_analyses
legs_GE_walk_users = legs_GE_walk_users.to_crs(operation_crs)

agglo_carreau = gpd.read_file(f'{output_file_path}GE/step-3/step3_aggregated_index_carreau200.gpkg') #from notebook 3_2
agglo_carreau = agglo_carreau.to_crs(operation_crs)


# VAGUE 1 : Mobilité

In [ ]:
# =============================================================================
# FILTRE OPTIONNEL — restreindre aux personnes séléctionnées via traces GPS
# =============================================================================

#walk_user_ids = legs_GE_walk_users["user_id_fors"].unique()
#PL_respondant_wave1 = PL_respondant_wave1[PL_respondant_wave1["id"].isin(walk_user_ids)]
#print(f"Filtre actif — répondants marcheurs GPS: {len(PL_respondant_wave1)}")

# =============================================================================

## DATAFRAME DES REPONDANTS DE LA VAGUE 1 - MOBILITE

In [ ]:
PL_respondant_wave1.head()

In [ ]:
# Spatial join: assign each area to a respondant

PL_respondant_wave1 = PL_respondant_wave1.drop(columns=["index_right", "AREA_TYPE"], errors="ignore") #clear the column 'index_right' avoiding to redo the spatial join
PL_respondant_wave1 = gpd.sjoin(PL_respondant_wave1, agglo_commune[["AREA_TYPE", "geometry"]], how="left", predicate="within")
PL_respondant_wave1 = PL_respondant_wave1.drop(columns=["index_right"], errors="ignore") #clear the column 'index_right' avoiding to redo the spatial join

In [ ]:
PL_respondant_wave1

In [ ]:
PL_respondant_wave1.dtypes

In [ ]:
n_total = PL_respondant_wave1['id'].nunique()
print(f"Total respondents: {n_total}")
#print(len(PL_respondant_wave1))

In [ ]:
# Afficher toutes les colonnes
pd.set_option('display.max_rows', None)
print(pd.DataFrame({"column": PL_respondant_wave1.columns, "dtype": PL_respondant_wave1.dtypes.values}))
pd.reset_option('display.max_rows')

## Study place

In [ ]:
# Q45_R = 9  → main mode to study place is walking
# Q46_8_R = 1 → combines another mode with walking

has_study = PL_respondant_wave1['Q45_R'].notna()
n_study = has_study.sum()

walk_study     = PL_respondant_wave1['Q45_R'] == 9
combined_study = PL_respondant_wave1['Q46_8_R'] == 1
any_walk_study = walk_study | combined_study

print(f"Respondents going to a study place:          {n_study}")
print(f"Walk as main mode       (Q45_R=9):           {walk_study.sum()}  ({walk_study.sum()/n_study*100:.1f}%)")
print(f"Walk combined           (Q46_8_R=1):         {combined_study.sum()}  ({combined_study.sum()/n_study*100:.1f}%)")
print(f"Walk at least partially (main OR combined):  {any_walk_study.sum()}  ({any_walk_study.sum()/n_study*100:.1f}%)")
print()
print("Q45_R — main mode to study place:")
labels = {1:'Multiple mode', 2:'Car', 3:'Car passenger', 4:'train', 5:'Public transport', 6:'Bicycle', 7:'Motorcycle', 8:'scooter', 9:'walk', 10:'boat', 11:'Other'}
print(PL_respondant_wave1.loc[has_study, 'Q45_R'].value_counts().sort_index().rename(labels))

print("\nWalk as main mode (Q45_R=9) by AREA_TYPE:")
area_study = (
    PL_respondant_wave1[has_study]
    .dropna(subset=["AREA_TYPE"])
    .groupby("AREA_TYPE")
    .apply(lambda g: pd.Series({
        "total": len(g),
        "walk_main": (g["Q45_R"] == 9).sum(),
        "walk_combined": (g["Q46_8_R"] == 1).sum(),
    }), include_groups=False)
    .assign(
        walk_main_pct=lambda x: (x["walk_main"] / x["total"] * 100).round(1),
        walk_combined_pct=lambda x: (x["walk_combined"] / x["total"] * 100).round(1),
    )
    .sort_values("walk_main_pct", ascending=False)
)
display(area_study)

In [ ]:
import plotly.express as px

area_study_melted = (
    area_study.reset_index()
    .melt(id_vars="AREA_TYPE", value_vars=["walk_main", "walk_combined"],
          var_name="mode", value_name="count")
    .replace({"walk_main": "Walk (main mode)", "walk_combined": "Walk (combined)"})
)
pct_map = area_study.reset_index().set_index("AREA_TYPE")[["walk_main_pct", "walk_combined_pct"]]
area_study_melted["pct"] = area_study_melted.apply(
    lambda r: pct_map.loc[r["AREA_TYPE"], "walk_main_pct" if r["mode"] == "Walk (main mode)" else "walk_combined_pct"], axis=1
)

fig = px.bar(
    area_study_melted,
    x="AREA_TYPE",
    y="count",
    color="mode",
    barmode="group",
    text=area_study_melted["pct"].astype(str) + "%",
    labels={"AREA_TYPE": "Area type", "count": "Respondents", "mode": "Walk type"},
    title="Q45 — Walk to study place by area type",
    color_discrete_sequence=px.colors.qualitative.Safe,
)
fig.update_traces(textposition="outside")
fig.update_layout(
    xaxis_tickangle=-30,
    yaxis_title="Nb of people",
    xaxis_title="Area type",
    height=500,
    legend_title="Mode",
)
fig.show()

## Work place

In [ ]:
# Q54_R = 9  → main mode to work place is walking
# Q55_8_R = 1 → combines another mode with walking

has_work = PL_respondant_wave1['Q54_R'].notna()
n_work = has_work.sum()

walk_work     = PL_respondant_wave1['Q54_R'] == 9
combined_work = PL_respondant_wave1['Q55_8_R'] == 1
any_walk_work = walk_work | combined_work

print(f"Respondents going to a work place:           {n_work}")
print(f"Walk as main mode       (Q54_R=9):           {walk_work.sum()}  ({walk_work.sum()/n_work*100:.1f}%)")
print(f"Walk combined           (Q55_8_R=1):         {combined_work.sum()}  ({combined_work.sum()/n_work*100:.1f}%)")
print(f"Walk at least partially (main OR combined):  {any_walk_work.sum()}  ({any_walk_work.sum()/n_work*100:.1f}%)")
print()
print("Q54_R — main mode to work place:")
labels = {1:'Multiple mode', 2:'Car', 3:'Car passenger', 4:'train', 5:'Public transport', 6:'Bicycle', 7:'Motorcycle', 8:'scooter', 9:'walk', 10:'boat', 11:'Other'}
print(PL_respondant_wave1.loc[has_work, 'Q54_R'].value_counts().sort_index().rename(labels))

print("\nWalk as main mode (Q54_R=9) by AREA_TYPE:")
area_work = (
    PL_respondant_wave1[has_work]
    .dropna(subset=["AREA_TYPE"])
    .groupby("AREA_TYPE")
    .apply(lambda g: pd.Series({
        "total": len(g),
        "walk_main": (g["Q54_R"] == 9).sum(),
        "walk_combined": (g["Q55_8_R"] == 1).sum(),
    }), include_groups=False)
    .assign(
        walk_main_pct=lambda x: (x["walk_main"] / x["total"] * 100).round(1),
        walk_combined_pct=lambda x: (x["walk_combined"] / x["total"] * 100).round(1),
    )
    .sort_values("walk_main_pct", ascending=False)
)
display(area_work)

In [ ]:
import plotly.express as px

area_work_melted = (
    area_work.reset_index()
    .melt(id_vars="AREA_TYPE", value_vars=["walk_main", "walk_combined"],
          var_name="mode", value_name="count")
    .replace({"walk_main": "Walk (main mode)", "walk_combined": "Walk (combined)"})
)
pct_map = area_work.reset_index().set_index("AREA_TYPE")[["walk_main_pct", "walk_combined_pct"]]
area_work_melted["pct"] = area_work_melted.apply(
    lambda r: pct_map.loc[r["AREA_TYPE"], "walk_main_pct" if r["mode"] == "Walk (main mode)" else "walk_combined_pct"], axis=1
)

fig = px.bar(
    area_work_melted,
    x="AREA_TYPE",
    y="count",
    color="mode",
    barmode="group",
    text=area_work_melted["pct"].astype(str) + "%",
    labels={"AREA_TYPE": "Area type", "count": "Respondents", "mode": "Walk type"},
    title="Q54 — Walk as mode to work place, by area type",
    color_discrete_sequence=px.colors.qualitative.Safe,
)
fig.update_traces(textposition="outside")
fig.update_layout(
    xaxis_tickangle=-30,
    yaxis_title="Nb of people",
    xaxis_title="Area type",
    height=500,
    legend_title="Mode",
)
fig.show()

## Summary table

In [ ]:
summary = pd.DataFrame({
    'Destination':             ['Study place', 'Work place'],
    'N respondents':           [n_study, n_work],
    'Walk (main mode)':        [walk_study.sum(), walk_work.sum()],
    'Walk (main) %':           [round(walk_study.sum()/n_study*100, 1), round(walk_work.sum()/n_work*100, 1)],
    'Walk (combined)':         [combined_study.sum(), combined_work.sum()],
    'Walk (combined) %':       [round(combined_study.sum()/n_study*100, 1), round(combined_work.sum()/n_work*100, 1)],
    'Walk (any)':              [any_walk_study.sum(), any_walk_work.sum()],
    'Walk (any) %':            [round(any_walk_study.sum()/n_study*100, 1), round(any_walk_work.sum()/n_work*100, 1)],
})
display(summary)

## Age groups

In [ ]:
# Current year for age calculation
CURRENT_YEAR = 2023 #date of the report wave 1

PL_respondant_wave1['age'] = CURRENT_YEAR - PL_respondant_wave1['date_naissance']

bins   = [18, 25, 35, 50, 65]
labels = ['18-24', '25-34', '35-49', '50-64']
PL_respondant_wave1['age_group'] = pd.cut(PL_respondant_wave1['age'], bins=bins, labels=labels, right=False)

age_stats = (
    PL_respondant_wave1['age_group']
    .value_counts()
    .sort_index()
    .rename_axis('Age group')
    .reset_index(name='Count')
)
age_stats['%'] = (age_stats['Count'] / age_stats['Count'].sum() * 100).round(1)

display(age_stats)
print(f"\nNot in any category (outside 18-64): {PL_respondant_wave1['age_group'].isna().sum()}")

## Upgrade/create walk paths

In [ ]:
# Q36_15 = 1 → respondent wants to upgrade/create walk paths in Greater Geneva

has_q36 = PL_respondant_wave1['Q36_15'].notna()
n_q36 = has_q36.sum()

want_walk = PL_respondant_wave1['Q36_15'] == 1

print(f"Total respondents (Q36 answered):             {n_q36}")
print(f"Want walk path improvements (Q36_15=1):       {want_walk.sum()}  ({want_walk.sum()/n_q36*100:.1f}%)")
print(f"Did not select this option  (Q36_15=0):       {(~want_walk & has_q36).sum()}  ({(~want_walk & has_q36).sum()/n_q36*100:.1f}%)")
print()
print("Breakdown by AREA_TYPE:")
area_stats = (
    PL_respondant_wave1
    .dropna(subset=["AREA_TYPE"])
    .groupby("AREA_TYPE")["Q36_15"]
    .agg(total="count", want=("sum"))
    .assign(pct=lambda x: (x["want"] / x["total"] * 100).round(1))
    .sort_values("pct", ascending=False)
)
display(area_stats)

In [ ]:
import plotly.express as px

# Count respondents who selected Q36=15 (want to upgrade/create walk paths), grouped by AREA_TYPE
q36_by_area = (
    PL_respondant_wave1
    .dropna(subset=["AREA_TYPE"])
    .groupby("AREA_TYPE")
    .agg(
        total=("Q36_15", "count"),
        want_walk_path=("Q36_15", "sum")
    )
    .reset_index()
)
q36_by_area["pct"] = (q36_by_area["want_walk_path"] / q36_by_area["total"] * 100).round(1)
q36_by_area = q36_by_area.sort_values("pct", ascending=False)

fig = px.bar(
    q36_by_area,
    x="AREA_TYPE",
    y="want_walk_path",
    text=q36_by_area["pct"].astype(str) + "%",
    labels={"AREA_TYPE": "Area type", "want_walk_path": "Respondents wanting walk path improvements"},
    title="Q36 — Respondents wanting to upgrade/create walk paths, by area type",
    color="AREA_TYPE",
    color_discrete_sequence=px.colors.qualitative.Safe,
)
fig.update_traces(textposition="outside")
fig.update_layout(
    xaxis_tickangle=-30,
    showlegend=False,
    yaxis_title="Count (Q36_15 = 1)",
    xaxis_title="Area type",
    height=500,
)
fig.show()

only 6.4% of poeple living in the major metro center (58/912) want to upgrade walk path

## Profil of people walking people

In [ ]:
n_total    = len(PL_respondant_wave1)
n_missing  = PL_respondant_wave1["Q120"].isna().sum()
n_dontknow = (PL_respondant_wave1["Q120"] == 10.0).sum()
n_answered = n_total - n_missing - n_dontknow

print(f"Total répondants:              {n_total}")
print(f"  Ont répondu (1-9):           {n_answered}  ({n_answered/n_total*100:.1f}%)")
print(f"  Ne sait pas / NR (10):       {n_dontknow}  ({n_dontknow/n_total*100:.1f}%)")
print(f"  Pas de réponse (NaN):        {n_missing}   ({n_missing/n_total*100:.1f}%)")

In [ ]:
income_labels = {
    1:  "< 2'000 CHF",
    2:  "2'000–4'000",
    3:  "4'001–6'000",
    4:  "6'001–8'000",
    5:  "8'001–10'000",
    6:  "10'001–12'000",
    7:  "12'001–14'000",
    8:  "14'001–16'000",
    9:  "> 16'000 CHF",
    10: "Ne sait pas / NR",
}
income_order = list(income_labels.values())

# Filtre Q46_8_R == 1
filter_q46 = PL_respondant_wave1[PL_respondant_wave1["Q46_8_R"] == 1.0].copy()
filter_q46["income_label"] = filter_q46["Q120"].map(income_labels)

# Filtre Q45_R == 9
filter_q45 = PL_respondant_wave1[PL_respondant_wave1["Q45_R"] == 9.0].copy()
filter_q45["income_label"] = filter_q45["Q120"].map(income_labels)

for df, title, color in [
    (filter_q46, "Revenus — répondants Q46_8_R = 1", "#4C78A8"),
    (filter_q45, "Revenus — répondants Q45_R = 9",   "#E45756"),
]:
    counts = (
        df["income_label"]
        .value_counts()
        .reindex(income_order)
        .fillna(0)
        .reset_index()
    )
    counts.columns = ["income_label", "count"]
    counts["pct"] = (counts["count"] / counts["count"].sum() * 100).round(1)

    print(f"\n{title} — n={len(df)}")
    display(counts)

    fig = px.bar(
        counts,
        x="income_label",
        y="count",
        text=counts["pct"].astype(str) + "%",
        title=f"{title} (n={len(df)})",
        labels={"income_label": "Revenu mensuel brut", "count": "Nombre de répondants"},
        color_discrete_sequence=[color],
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(
        xaxis_tickangle=-30,
        height=450,
        yaxis_title="Nombre de répondants",
        xaxis_title="Revenu mensuel brut du ménage",
    )
    fig.show()

In [ ]:
# Filtre Q54_R == 9 (uniquement marche à pied pour aller au travail)
filter_q54 = PL_respondant_wave1[PL_respondant_wave1["Q54_R"] == 9.0].copy()
filter_q54["income_label"] = filter_q54["Q120"].map(income_labels)

# Filtre Q55_8_R == 1 (marche à pied fait partie des modes pour aller au travail)
filter_q55 = PL_respondant_wave1[PL_respondant_wave1["Q55_8_R"] == 1.0].copy()
filter_q55["income_label"] = filter_q55["Q120"].map(income_labels)

for df, title, color in [
    (filter_q54, "Revenus — répondants Q54_R = 9 (uniquement marche au travail)",      "#4C78A8"),
    (filter_q55, "Revenus — répondants Q55_8_R = 1 (marche incluse modes au travail)", "#E45756"),
]:
    counts = (
        df["income_label"]
        .value_counts()
        .reindex(income_order)
        .fillna(0)
        .reset_index()
    )
    counts.columns = ["income_label", "count"]
    counts["pct"] = (counts["count"] / counts["count"].sum() * 100).round(1)

    print(f"\n{title} — n={len(df)}")
    display(counts)

    fig = px.bar(
        counts,
        x="income_label",
        y="count",
        text=counts["pct"].astype(str) + "%",
        title=f"{title} (n={len(df)})",
        labels={"income_label": "Revenu mensuel brut", "count": "Nombre de répondants"},
        color_discrete_sequence=[color],
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(
        xaxis_tickangle=-30,
        height=450,
        yaxis_title="Nombre de répondants",
        xaxis_title="Revenu mensuel brut du ménage",
    )
    fig.show()

### Mode de transport en fonction du revenus (lieu de formation)

In [ ]:
transport_labels = {
    1:  "Multimodal",
    2:  "Voiture (conducteur)",
    3:  "Voiture (passager)",
    4:  "Train",
    5:  "TP (sans train)",
    6:  "Vélo",
    7:  "Moto",
    8:  "Scooter / cyclomoteur",
    9:  "Marche à pied",
    10: "Bateau",
    11: "Autre mode unique",
}

cross_df = PL_respondant_wave1[["Q45_R", "Q120"]].dropna().copy()
cross_df["transport_label"] = cross_df["Q45_R"].map(transport_labels)
cross_df["income_label"]    = cross_df["Q120"].map(income_labels)

print(f"Répondants à la question avec un salaire renseigné: {len(cross_df)}")

# Tableau croisé : lignes = revenu, colonnes = mode de transport
cross_table = (
    cross_df
    .groupby(["income_label", "transport_label"])
    .size()
    .unstack(fill_value=0)
    .reindex(income_order)
)

# Version en % par ligne (= pour chaque tranche de revenu, répartition des modes)
cross_table_pct = cross_table.div(cross_table.sum(axis=1), axis=0).mul(100).round(1)

print("Tableau croisé Q45_R × Q120 (% par tranche de revenu):")
display(cross_table_pct)

# Heatmap
fig = px.imshow(
    cross_table_pct,
    text_auto=True,
    color_continuous_scale="Blues",
    title="Mode de transport principal × Revenu mensuel du ménage (%) - Lieu de formation",
    labels={"x": "Mode de transport", "y": "Revenu mensuel", "color": "%"},
    aspect="auto",
)
fig.update_layout(height=500, xaxis_tickangle=-30)
fig.show()

# Stacked bar par tranche de revenu
cross_plot = cross_table_pct.reset_index().melt(
    id_vars="income_label",
    var_name="transport_label",
    value_name="pct"
)

fig = px.bar(
    cross_plot,
    x="income_label",
    y="pct",
    color="transport_label",
    barmode="stack",
    title="Répartition des modes de transport par tranche de revenu",
    labels={
        "income_label":    "Revenu mensuel du ménage",
        "pct":             "% des répondants",
        "transport_label": "Mode de transport",
    },
)
fig.update_layout(
    xaxis_tickangle=-30,
    height=500,
    yaxis_title="% des répondants",
    legend_title="Mode de transport",
)
fig.show()

### Moyen de transport pour aller sur son lieu de travail 

In [ ]:
transport_labels = {
    1:  "Multimodal",
    2:  "Voiture (conducteur)",
    3:  "Voiture (passager)",
    4:  "Train",
    5:  "TP (sans train)",
    6:  "Vélo",
    7:  "Moto",
    8:  "Scooter / cyclomoteur",
    9:  "Marche à pied",
    10: "Bateau",
    11: "Autre mode unique",
}

cross_df = PL_respondant_wave1[["Q54_R", "Q120"]].dropna().copy()
cross_df["transport_label"] = cross_df["Q54_R"].map(transport_labels)
cross_df["income_label"]    = cross_df["Q120"].map(income_labels)

print(f"Répondants à la question avec un salair renseigné: {len(cross_df)}")


# Tableau croisé : lignes = revenu, colonnes = mode de transport
cross_table = (
    cross_df
    .groupby(["income_label", "transport_label"])
    .size()
    .unstack(fill_value=0)
    .reindex(income_order)
)

# Version en % par ligne (= pour chaque tranche de revenu, répartition des modes)
cross_table_pct = cross_table.div(cross_table.sum(axis=1), axis=0).mul(100).round(1)

print("Tableau croisé Q54_R × Q120 (% par tranche de revenu):")
display(cross_table_pct)

# Heatmap
fig = px.imshow(
    cross_table_pct,
    text_auto=True,
    color_continuous_scale="Blues",
    title="Mode de transport principal × Revenu mensuel du ménage (%) - Lieu de travail",
    labels={"x": "Mode de transport", "y": "Revenu mensuel", "color": "%"},
    aspect="auto",
)
fig.update_layout(height=500, xaxis_tickangle=-30)
fig.show()

# Stacked bar par tranche de revenu
cross_plot = cross_table_pct.reset_index().melt(
    id_vars="income_label",
    var_name="transport_label",
    value_name="pct"
)

fig = px.bar(
    cross_plot,
    x="income_label",
    y="pct",
    color="transport_label",
    barmode="stack",
    title="Répartition des modes de transport par tranche de revenu",
    labels={
        "income_label":    "Revenu mensuel du ménage",
        "pct":             "% des répondants",
        "transport_label": "Mode de transport",
    },
)
fig.update_layout(
    xaxis_tickangle=-30,
    height=500,
    yaxis_title="% des répondants",
    legend_title="Mode de transport",
)
fig.show()

### Parmis les multimodaux qui vont sur leur lieu de formation

In [ ]:
mode_cols = {
    "Q46_1_R": "Voiture (conducteur)",
    "Q46_2_R": "Voiture (passager)",
    "Q46_3_R": "Train",
    "Q46_4_R": "TP (sans train)",
    "Q46_5_R": "Vélo",
    "Q46_6_R": "Moto",
    "Q46_7_R": "Scooter / cyclomoteur",
    "Q46_8_R": "Marche à pied",
    "Q46_9_R": "Bateau",
    "Q46_10_R": "Autre",
}

# Filtrer uniquement les personnes multimodales (Q45_R == 1)
multimodal = PL_respondant_wave1[PL_respondant_wave1["Q45_R"] == 1].copy()
multimodal["income_label"] = multimodal["Q120"].map(income_labels)

print(f"Répondants multimodaux: {len(multimodal)}")
print(f"  dont avec revenu renseigné: {multimodal['income_label'].notna().sum()}")

# --- Tableau : % d'utilisation de chaque mode par tranche de revenu ---
rows = []
for col, mode_name in mode_cols.items():
    if col in multimodal.columns:
        for income, group in multimodal.dropna(subset=["income_label"]).groupby("income_label"):
            n_total = len(group)
            n_use   = (group[col] == 1.0).sum()
            rows.append({
                "income_label": income,
                "mode":         mode_name,
                "n_use":        n_use,
                "pct":          round(n_use / n_total * 100, 1) if n_total > 0 else 0,
            })

multimodal_df = pd.DataFrame(rows)

# Tableau croisé revenu × mode
cross_multimodal = (
    multimodal_df
    .pivot(index="income_label", columns="mode", values="pct")
    .reindex(income_order)
)

print("\n% d'utilisation de chaque mode par tranche de revenu (multimodaux uniquement):")
display(cross_multimodal)

# --- Heatmap ---
fig = px.imshow(
    cross_multimodal,
    text_auto=True,
    color_continuous_scale="Blues",
    title="Modes de transport combinés × Revenu mensuel (multimodaux, %)",
    labels={"x": "Mode de transport", "y": "Revenu mensuel", "color": "%"},
    aspect="auto",
)
fig.update_layout(height=500, xaxis_tickangle=-30)
fig.show()

# --- Stacked bar ---
cross_plot = multimodal_df.dropna(subset=["income_label"])
cross_plot = cross_plot[cross_plot["income_label"].isin(income_order)]
cross_plot["income_label"] = pd.Categorical(cross_plot["income_label"], categories=income_order, ordered=True)
cross_plot = cross_plot.sort_values("income_label")

fig = px.bar(
    cross_plot,
    x="income_label",
    y="pct",
    color="mode",
    barmode="group",
    title="Modes de transport combinés par tranche de revenu (multimodaux) - Lieu de Formation",
    labels={
        "income_label": "Revenu mensuel du ménage",
        "pct":          "% des répondants utilisant ce mode",
        "mode":         "Mode de transport",
    },
)
fig.update_layout(
    xaxis_tickangle=-30,
    height=500,
    yaxis_title="% des répondants",
    legend_title="Mode de transport",
)
fig.show()

Par exemple: Marche à pied 45% pour la tranche de tel revenus -> 45% des personnes multimodales gagant cette tranche de prix là inclunt la rmarche à pied dans leur combinaisons de mode pour se rendre sur leur lieu de travail

### Multimodaux qui vont sur leur lieu de travail

In [ ]:
mode_cols = {
    "Q55_1_R": "Voiture (conducteur)",
    "Q55_2_R": "Voiture (passager)",
    "Q55_3_R": "Train",
    "Q55_4_R": "TP (sans train)",
    "Q55_5_R": "Vélo",
    "Q55_6_R": "Moto",
    "Q55_7_R": "Scooter / cyclomoteur",
    "Q55_8_R": "Marche à pied",
    "Q55_9_R": "Bateau",
    "Q55_10_R": "Autre",
}

# Filtrer uniquement les personnes multimodales (Q54_R == 1)
multimodal = PL_respondant_wave1[PL_respondant_wave1["Q54_R"] == 1].copy()
multimodal["income_label"] = multimodal["Q120"].map(income_labels)

print(f"Répondants multimodaux: {len(multimodal)}")
print(f"  dont avec revenu renseigné: {multimodal['income_label'].notna().sum()}")

# --- Tableau : % d'utilisation de chaque mode par tranche de revenu ---
rows = []
for col, mode_name in mode_cols.items():
    if col in multimodal.columns:
        for income, group in multimodal.dropna(subset=["income_label"]).groupby("income_label"):
            n_total = len(group)
            n_use   = (group[col] == 1.0).sum()
            rows.append({
                "income_label": income,
                "mode":         mode_name,
                "n_use":        n_use,
                "pct":          round(n_use / n_total * 100, 1) if n_total > 0 else 0,
            })

multimodal_df = pd.DataFrame(rows)

# Tableau croisé revenu × mode
cross_multimodal = (
    multimodal_df
    .pivot(index="income_label", columns="mode", values="pct")
    .reindex(income_order)
)

print("\n% d'utilisation de chaque mode par tranche de revenu (multimodaux uniquement):")
display(cross_multimodal)

# --- Heatmap ---
fig = px.imshow(
    cross_multimodal,
    text_auto=True,
    color_continuous_scale="Blues",
    title="Modes de transport combinés × Revenu mensuel (multimodaux, %)",
    labels={"x": "Mode de transport", "y": "Revenu mensuel", "color": "%"},
    aspect="auto",
)
fig.update_layout(height=500, xaxis_tickangle=-30)
fig.show()

# --- Stacked bar ---
cross_plot = multimodal_df.dropna(subset=["income_label"])
cross_plot = cross_plot[cross_plot["income_label"].isin(income_order)]
cross_plot["income_label"] = pd.Categorical(cross_plot["income_label"], categories=income_order, ordered=True)
cross_plot = cross_plot.sort_values("income_label")

fig = px.bar(
    cross_plot,
    x="income_label",
    y="pct",
    color="mode",
    barmode="group",
    title="Modes de transport combinés par tranche de revenu (multimodaux) - Lieu de Formation",
    labels={
        "income_label": "Revenu mensuel du ménage",
        "pct":          "% des répondants utilisant ce mode",
        "mode":         "Mode de transport",
    },
)
fig.update_layout(
    xaxis_tickangle=-30,
    height=500,
    yaxis_title="% des répondants",
    legend_title="Mode de transport",
)
fig.show()

# VAGUE RYTHME — Security & insecurity spots

In [ ]:
# Load all rythme respondents (respondent level, for general statistics)
PL_respondant_rythme = pd.read_csv(f'{input_file_path_PL}WAVE_RYTHM/OUTPUT_R/respondants_rythme_all.csv')

print(f"Total rythme respondents:          {len(PL_respondant_rythme)}")
print(f"  gave an insecurity spot:         {PL_respondant_rythme['has_insecurity'].sum()}  ({PL_respondant_rythme['has_insecurity'].mean()*100:.1f}%)")
print(f"  gave a security spot:            {PL_respondant_rythme['has_security'].sum()}  ({PL_respondant_rythme['has_security'].mean()*100:.1f}%)")
print(f"  gave both:                       {(PL_respondant_rythme['has_insecurity'] & PL_respondant_rythme['has_security']).sum()}")
print(f"  gave neither:                    {(~PL_respondant_rythme['has_insecurity'] & ~PL_respondant_rythme['has_security']).sum()}")
print("\n People can gave only one spot per category")

# Load spots (spatial, only those with coordinates within Geneva)

# Load spots (spatial, only those with coordinates within Geneva)
PL_spots_insecurity = gpd.read_file(f'{input_file_path_PL}WAVE_RYTHM/OUTPUT_R/spots_insecurity.gpkg').to_crs(operation_crs)
PL_spots_security   = gpd.read_file(f'{input_file_path_PL}WAVE_RYTHM/OUTPUT_R/spots_security.gpkg').to_crs(operation_crs)

# Spatial join: assign AREA_TYPE (agglo_commune), GIREC zone, and commune to each spot
for spots in [PL_spots_insecurity, PL_spots_security]:
    spots.drop(columns=["index_right", "AREA_TYPE"], errors="ignore", inplace=True)

# agglo_commune → AREA_TYPE
PL_spots_insecurity = gpd.sjoin(PL_spots_insecurity, agglo_commune[["AREA_TYPE", "geometry"]], how="left", predicate="within")
PL_spots_insecurity = PL_spots_insecurity.drop(columns=["index_right"], errors="ignore")

PL_spots_security   = gpd.sjoin(PL_spots_security, agglo_commune[["AREA_TYPE", "geometry"]], how="left", predicate="within")
PL_spots_security   = PL_spots_security.drop(columns=["index_right"], errors="ignore")

# zones_girec
girec_col = "NOM"
PL_spots_insecurity = gpd.sjoin(PL_spots_insecurity, zones_girec[[girec_col, "geometry"]], how="left", predicate="within")
PL_spots_insecurity = PL_spots_insecurity.drop(columns=["index_right"], errors="ignore")
PL_spots_insecurity = PL_spots_insecurity.rename(columns={girec_col: "sous_secteur"})

PL_spots_security   = gpd.sjoin(PL_spots_security, zones_girec[[girec_col, "geometry"]], how="left", predicate="within")
PL_spots_security   = PL_spots_security.drop(columns=["index_right"], errors="ignore")
PL_spots_security = PL_spots_security.rename(columns={girec_col: "sous_secteur"})

# commune_GE 
commune_col = "COMMUNE"  
PL_spots_insecurity = gpd.sjoin(PL_spots_insecurity, commune_GE[[commune_col, "geometry"]], how="left", predicate="within")
PL_spots_insecurity = PL_spots_insecurity.drop(columns=["index_right"], errors="ignore")

PL_spots_security   = gpd.sjoin(PL_spots_security, commune_GE[[commune_col, "geometry"]], how="left", predicate="within")
PL_spots_security   = PL_spots_security.drop(columns=["index_right"], errors="ignore")

print(f"\nInsecurity spots in Geneva (spatial): {len(PL_spots_insecurity)}")
print(f"Security spots in Geneva (spatial):   {len(PL_spots_security)}")
print(f"Columns: {PL_spots_insecurity.columns.tolist()}")

PL_spots_insecurity.head()

In [ ]:
# Manual correction for respondents outside agglo_commune polygons
#corrigé manuellement parce que ces points tombaient hors de la zone des communes, ils tombaient dans le lac, mais étaient des erreur géographique car faisaient référence à des lieux sur les rives ou autre endroits proches, donc on les gardes

missing_ids_insecurity = ["CH10980", "CH22147", "FR8225", "CH11288", "CH2979", "CH15784", "FR14573"]
missing_ids_security   = ["CH10980", "CH22147", "FR8225", "CH11288", "CH2979", "CH15784", "FR14573"]

PL_spots_insecurity.loc[PL_spots_insecurity["id"].isin(missing_ids_insecurity), "AREA_TYPE"] = "major metro centers"
PL_spots_security.loc[PL_spots_security["id"].isin(missing_ids_security),       "AREA_TYPE"] = "major metro centers"

In [ ]:
# Vérifier quels IDs de la liste sont présents dans chaque fichier
print("Dans insecurity:", PL_spots_insecurity[PL_spots_insecurity["id"].isin(missing_ids_insecurity)][["id", "AREA_TYPE"]])
print("Dans security:",   PL_spots_security[PL_spots_security["id"].isin(missing_ids_security)][["id", "AREA_TYPE"]])

In [ ]:
print("Colonnes dupliquées insecurity:", PL_spots_insecurity.columns[PL_spots_insecurity.columns.duplicated()].tolist())
print("Colonnes dupliquées security:",   PL_spots_security.columns[PL_spots_security.columns.duplicated()].tolist())
print("\nToutes les colonnes insecurity:", PL_spots_insecurity.columns.tolist())

In [ ]:
# Overall statistics
n_insecurity = len(PL_spots_insecurity)
n_security   = len(PL_spots_security)
n_spots      = n_insecurity + n_security

print(f"Total spots:       {n_spots}")
print(f"  Insecurity:      {n_insecurity}  ({n_insecurity/n_spots*100:.1f}%)")
print(f"  Security:        {n_security}    ({n_security/n_spots*100:.1f}%)")
print(f"  Missing AREA_TYPE (insecurity): {PL_spots_insecurity['AREA_TYPE'].isna().sum()}")
print(f"  Missing AREA_TYPE (security):   {PL_spots_security['AREA_TYPE'].isna().sum()}")
print()

# Breakdown by AREA_TYPE
print("Breakdown by AREA_TYPE:")

# On recombine temporairement juste pour ce tableau récapitulatif
PL_spots_combined = pd.concat([
    PL_spots_insecurity.assign(spot_type="insecurity"),
    PL_spots_security.assign(spot_type="security")
], ignore_index=True)

area_spots = (
    PL_spots_combined
    .dropna(subset=["AREA_TYPE"])
    .groupby(["AREA_TYPE", "spot_type"])
    .size()
    .unstack(fill_value=0)
    .assign(total=lambda x: x.sum(axis=1))
    .assign(
        insecurity_pct=lambda x: (x.get("insecurity", 0) / x["total"] * 100).round(1),
        security_pct=  lambda x: (x.get("security",   0) / x["total"] * 100).round(1),
    )
    .sort_values("total", ascending=False)
)
display(area_spots)

In [ ]:
# Gender breakdown
gender_map = {"1": "Man", "2": "Woman", "3": "Other", "4": "Other", "5": "Other"}

PL_spots_insecurity["gender"] = PL_spots_insecurity["P0_genre"].astype(str).map(gender_map)
PL_spots_security["gender"]   = PL_spots_security["P0_genre"].astype(str).map(gender_map)

# Tableau croisé gender × spot_type (recombine temporairement)
PL_spots_combined = pd.concat([
    PL_spots_insecurity.assign(spot_type="insecurity"),
    PL_spots_security.assign(spot_type="security")
])

print("Security & insecurity spots by gender:")
gender_spots = (
    PL_spots_combined
    .groupby(["gender", "spot_type"])
    .size()
    .unstack(fill_value=0)
    .assign(total=lambda x: x.sum(axis=1))
    .assign(
        insecurity_pct=lambda x: (x.get("insecurity", 0) / x["total"] * 100).round(1),
        security_pct=  lambda x: (x.get("security",   0) / x["total"] * 100).round(1),
    )
    .sort_values("total", ascending=False)
)
display(gender_spots)

print("\nInsecurity spots by gender (% of all insecurity spots):")
insec_by_gender = PL_spots_insecurity["gender"].value_counts()
print((insec_by_gender / insec_by_gender.sum() * 100).round(1).rename("% of insecurity spots"))

print("\nSecurity spots by gender (% of all security spots):")
sec_by_gender = PL_spots_security["gender"].value_counts()
print((sec_by_gender / sec_by_gender.sum() * 100).round(1).rename("% of security spots"))

In [ ]:
gender_plot = (
    PL_spots_combined
    .groupby(["gender", "spot_type"])
    .size()
    .reset_index(name="count")
)
gender_plot["total"] = gender_plot.groupby("gender")["count"].transform("sum")
gender_plot["pct"] = (gender_plot["count"] / gender_plot["total"] * 100).round(1)

fig = px.bar(
    gender_plot,
    x="gender",
    y="count",
    color="spot_type",
    barmode="group",
    text=gender_plot["pct"].astype(str) + "%",
    labels={"gender": "Gender", "count": "Number of spots", "spot_type": "Spot type"},
    title="Rythme — Security & insecurity spots by gender",
    color_discrete_map={"insecurity": "#E45756", "security": "#4C78A8"},
    category_orders={"gender": ["Man", "Woman", "Other"]},
)
fig.update_traces(textposition="outside")
fig.update_layout(
    yaxis_title="Number of spots",
    xaxis_title="Gender",
    height=500,
    legend_title="Spot type",
)
fig.show()

In [ ]:
import plotly.express as px

area_spots_plot = (
    PL_spots_combined
    .dropna(subset=["AREA_TYPE"])
    .groupby(["AREA_TYPE", "spot_type"])
    .size()
    .reset_index(name="count")
)
area_spots_plot["total"] = area_spots_plot.groupby("AREA_TYPE")["count"].transform("sum")
area_spots_plot["pct"] = (area_spots_plot["count"] / area_spots_plot["total"] * 100).round(1)

fig = px.bar(
    area_spots_plot,
    x="AREA_TYPE",
    y="count",
    color="spot_type",
    barmode="group",
    text=area_spots_plot["pct"].astype(str) + "%",
    labels={"AREA_TYPE": "Area type", "count": "Number of spots", "spot_type": "Spot type"},
    title="Rythme — Security & insecurity spots by area type",
    color_discrete_map={"insecurity": "#E45756", "security": "#4C78A8"},
)
fig.update_traces(textposition="outside")
fig.update_layout(
    xaxis_tickangle=-30,
    yaxis_title="Number of spots",
    xaxis_title="Area type",
    height=500,
    legend_title="Spot type",
)
fig.show()

## Join with walkability index (nearest segment)

In [ ]:
print(f"Segments loaded: {len(step3_index)}")
print(f"Columns: {step3_index.columns.tolist()}")

# Nearest segment join: each spot gets the attributes of its closest segment
PL_spots_insecurity = gpd.sjoin_nearest(
    PL_spots_insecurity,
    step3_index,
    how="left",
    distance_col="dist_to_segment"
)
PL_spots_insecurity = PL_spots_insecurity.drop(columns=["index_right"], errors="ignore")

PL_spots_security = gpd.sjoin_nearest(
    PL_spots_security,
    step3_index,
    how="left",
    distance_col="dist_to_segment"
)
PL_spots_security = PL_spots_security.drop(columns=["index_right"], errors="ignore")

print(f"\nInsecurity spots after join: {len(PL_spots_insecurity)}")
print(f"  Mean distance to nearest segment: {PL_spots_insecurity['dist_to_segment'].mean():.1f} m")
print(f"  Max distance to nearest segment:  {PL_spots_insecurity['dist_to_segment'].max():.1f} m")

print(f"\nSecurity spots after join: {len(PL_spots_security)}")
print(f"  Mean distance to nearest segment: {PL_spots_security['dist_to_segment'].mean():.1f} m")
print(f"  Max distance to nearest segment:  {PL_spots_security['dist_to_segment'].max():.1f} m")

In [ ]:
import plotly.express as px

for label, df in [("Insecurity", PL_spots_insecurity), ("Security", PL_spots_security)]:
    print(f"\n{label} spots — distance to nearest segment:")
    print(df["dist_to_segment"].describe().round(1))
    
    fig = px.histogram(
        df,
        x="dist_to_segment",
        nbins=50,
        title=f"Rythme — {label} spots: distance to nearest walkability segment",
        labels={"dist_to_segment": "Distance (m)", "count": "Number of spots"},
        color_discrete_sequence=["#E45756" if label == "Insecurity" else "#4C78A8"],
    )
    fig.update_layout(
        xaxis_title="Distance to nearest segment (m)",
        yaxis_title="Number of spots",
        height=400,
    )
    fig.show()

# Cumulative distribution pour mieux voir les seuils
PL_spots_combined_dist = pd.concat([
    PL_spots_insecurity.assign(spot_type="insecurity"),
    PL_spots_security.assign(spot_type="security")
])

print("\nPercentiles de distance (tous spots):")
percentiles = [50, 75, 90, 95, 99, 100]
for p in percentiles:
    val = PL_spots_combined_dist["dist_to_segment"].quantile(p/100)
    print(f"  p{p:3d}: {val:.1f} m")

In [ ]:
# Compute p95 threshold from all spots combined
p95_threshold = PL_spots_combined_dist["dist_to_segment"].quantile(0.95)
print(f"P95 threshold: {p95_threshold:.1f} m")

# Apply threshold: walk_index set to NaN if distance exceeds p95
for label, df in [("Insecurity", PL_spots_insecurity), ("Security", PL_spots_security)]:
    total = len(df)
    n_clipped = (df["dist_to_segment"] > p95_threshold).sum()
    n_kept    = total - n_clipped
    
    print(f"\n{label} spots:")
    print(f"  Total:           {total}")
    print(f"  Kept (≤ {p95_threshold:.1f} m): {n_kept}  ({n_kept/total*100:.1f}%)")
    print(f"  Clipped (> {p95_threshold:.1f} m): {n_clipped}  ({n_clipped/total*100:.1f}%)")

# Apply
PL_spots_insecurity["walk_index"] = PL_spots_insecurity["walk_index"].where(
    PL_spots_insecurity["dist_to_segment"] <= p95_threshold, other=np.nan
)
PL_spots_security["walk_index"] = PL_spots_security["walk_index"].where(
    PL_spots_security["dist_to_segment"] <= p95_threshold, other=np.nan
)

print(f"\nWalk index NaN after clipping:")
print(f"  Insecurity: {PL_spots_insecurity['walk_index'].isna().sum()}")
print(f"  Security:   {PL_spots_security['walk_index'].isna().sum()}")

In [ ]:
print(PL_spots_insecurity.columns.tolist())

### 1. Mean comparison by spot type

In [ ]:
#On doit recréer PL_spots_combined car le threshold sur les distances d'assignement a été appliqué séparement sur les fichiers

PL_spots_combined = pd.concat([
    PL_spots_insecurity.assign(spot_type="insecurity"),
    PL_spots_security.assign(spot_type="security")
])

stats = (
    PL_spots_combined
    .groupby("spot_type")[["walk_index", "Classe_Sécurité"]]
    .agg(["mean", "median", "std"])
    .round(3)
)
display(stats)

### 2. Distributions — violin plots

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["walk_index (general)", "Classe_Sécurité (security sub-index)"])

for col_idx, col in enumerate(["walk_index", "Classe_Sécurité"], start=1):
    fig.add_trace(go.Violin(
        y=PL_spots_insecurity[col].dropna(), name="insecurity", legendgroup="insecurity",
        showlegend=(col_idx == 1), line_color="#E45756", box_visible=True, meanline_visible=True,
    ), row=1, col=col_idx)
    fig.add_trace(go.Violin(
        y=PL_spots_security[col].dropna(), name="security", legendgroup="security",
        showlegend=(col_idx == 1), line_color="#4C78A8", box_visible=True, meanline_visible=True,
    ), row=1, col=col_idx)

fig.update_layout(
    title="Distribution of walk_index and Classe_Sécurité by spot type",
    violinmode="group",
    height=500,
    legend_title="Spot type",
)
fig.show()

### 3. walk_index vs Classe_Sécurité gap by spot type

In [ ]:
# Melt both indices into one column for side-by-side comparison
gap_df = (
    PL_spots_combined[["spot_type", "walk_index", "Classe_Sécurité"]]
    .melt(id_vars="spot_type", var_name="index", value_name="value")
)

gap_means = gap_df.groupby(["spot_type", "index"])["value"].mean().reset_index()
gap_means["value"] = gap_means["value"].round(3)

print("Mean values:")
display(gap_means.pivot(index="spot_type", columns="index", values="value")
        .assign(gap=lambda x: x["walk_index"] - x["Classe_Sécurité"])
        .round(3))

fig = px.bar(
    gap_means,
    x="index", y="value", color="spot_type",
    barmode="group",
    text="value",
    title="Mean walk_index vs Classe_Sécurité by spot type",
    color_discrete_map={"insecurity": "#E45756", "security": "#4C78A8"},
    labels={"index": "Index", "value": "Mean value", "spot_type": "Spot type"},
)
fig.update_traces(
    textposition="outside",
    width=0.15,  # ← largeur des barres (entre 0 et 1)
)
fig.update_layout(
    height=450,
    yaxis_title="Mean value",
    legend_title="Spot type",
    bargap=0.4,       # ← espace entre les groupes
    bargroupgap=0, # ← espace entre les barres d'un même groupe
)
fig.show()

### 4. By AREA_TYPE

In [ ]:
area_index = (
    PL_spots_combined
    .dropna(subset=["AREA_TYPE"])
    .groupby(["AREA_TYPE", "spot_type"])[["walk_index", "Classe_Sécurité"]]
    .mean()
    .round(3)
    .reset_index()
)

for idx_col in ["walk_index", "Classe_Sécurité"]:
    fig = px.bar(
        area_index,
        x="AREA_TYPE", y=idx_col, color="spot_type",
        barmode="group",
        text=area_index[idx_col].round(2),
        title=f"Mean {idx_col} by AREA_TYPE and spot type",
        color_discrete_map={"insecurity": "#E45756", "security": "#4C78A8"},
        labels={"AREA_TYPE": "Area type", idx_col: f"Mean {idx_col}", "spot_type": "Spot type"},
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(xaxis_tickangle=-30, height=450, legend_title="Spot type")
    fig.show()

### 5. Mann-Whitney U test (statistical significance)

In [ ]:
from scipy import stats as scipy_stats

for col in ["walk_index", "Classe_Sécurité"]:
    a = PL_spots_insecurity[col].dropna()
    b = PL_spots_security[col].dropna()
    stat, p = scipy_stats.mannwhitneyu(a, b, alternative="two-sided")

    print(f"--- {col} ---")
    print(f"  insecurity: n={len(a)}, median={a.median():.3f}")
    print(f"  security:   n={len(b)}, median={b.median():.3f}")
    print(f"  Mann-Whitney U={stat:.0f}, p={p:.4f}", end="  ")
    if p < 0.001:
        print("*** (p < 0.001)")
    elif p < 0.01:
        print("**  (p < 0.01)")
    elif p < 0.05:
        print("*   (p < 0.05)")
    else:
        print("(not significant)")
    print()

### Carte sécurité/insécurité par SOUS-SECTEUR

In [ ]:
# --- Enrichissement zones_girec avec walk_index (moyenne pondérée par longueur de segment) ---

# Comptages insécurité et sécurité par sous-secteur
counts_insecurity = (
    PL_spots_insecurity
    .groupby("sous_secteur")
    .size()
    .reset_index(name="n_insecurity_spots")
)

counts_security = (
    PL_spots_security
    .groupby("sous_secteur")
    .size()
    .reset_index(name="n_security_spots")
)

# --- Merge comptages spots ---
zones_girec_map = zones_girec.merge(
    counts_insecurity.rename(columns={"sous_secteur": "NOM"}),
    on="NOM", how="left"
).merge(
    counts_security.rename(columns={"sous_secteur": "NOM"}),
    on="NOM", how="left"
)

zones_girec_map["n_insecurity_spots"] = zones_girec_map["n_insecurity_spots"].fillna(0).astype(int)
zones_girec_map["n_security_spots"]   = zones_girec_map["n_security_spots"].fillna(0).astype(int)
zones_girec_map["n_total_spots"]      = zones_girec_map["n_insecurity_spots"] + zones_girec_map["n_security_spots"]
zones_girec_map["security_ratio"]     = (
    zones_girec_map["n_security_spots"] / zones_girec_map["n_total_spots"].replace(0, np.nan)
).round(3)  # NaN si aucun spot, entre 0 et 1 sinon

print("\n Top 10 des sous secteurs les moins sûres")
zones_girec_map[["NOM", "n_insecurity_spots", "n_security_spots", "n_total_spots", "security_ratio"]].sort_values("n_insecurity_spots", ascending=False).head(10)

In [ ]:
zones_girec_map

In [ ]:
print("Top 10 des sous secteurs les plus sûres")
zones_girec_map[["NOM", "n_insecurity_spots", "n_security_spots", "n_total_spots"]].sort_values("n_security_spots", ascending=False).head(10)

In [ ]:
zones_girec_map

In [ ]:
def plot_spots_map(gdf, col, cmap, title, label):
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Zones à 0 en gris clair
    gdf[gdf[col] == 0].plot(
        ax=ax, color="lightgrey", edgecolor=(0.5, 0.5, 0.5, 0.3), linewidth=0.5
    )
    
    # Zones > 0 avec gradient continu
    non_zero = gdf[gdf[col] > 0]
    non_zero.plot(
        column=col,
        cmap=cmap,
        legend=True,
        legend_kwds={"label": label},
        edgecolor=(0.5, 0.5, 0.5, 0.3),
        linewidth=0.5,
        vmin=1,  # gradient commence à 1 et non à 0
        vmax=non_zero[col].max(),
        ax=ax
    )
    
    # Ajouter gris dans la légende
    grey_patch = mpatches.Patch(color="lightgrey", label="0 spot")
    ax.legend(handles=[grey_patch], loc="lower left", framealpha=0.9)
    
    ax.set_title(title, fontsize=14)
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()

plot_spots_map(zones_girec_map, "n_insecurity_spots", "Reds",
               "Spots d'insécurité par sous-secteur GIREC", "Spots insécurité")

plot_spots_map(zones_girec_map, "n_security_spots", "Greens",
               "Spots de sécurité par sous-secteur GIREC", "Spots sécurité")

In [ ]:
fig = px.scatter(
    zones_girec_map,
    x="n_insecurity_spots",
    y="n_security_spots",
    text="NOM",
    title="Distribution des spots de sécurité et d'insécurité par sous-secteur GIREC",
    labels={
        "n_insecurity_spots": "Spots d'insécurité",
        "n_security_spots":   "Spots de sécurité",
    },
    hover_data=["NOM", "n_total_spots"],
)
fig.update_traces(textposition="top center", textfont_size=8)
fig.update_layout(height=600)
fig.show()

In [ ]:
from scipy.spatial.distance import cdist

fig, ax = plt.subplots(figsize=(12, 5))

scatter = ax.scatter(
    zones_girec_map["n_insecurity_spots"],
    zones_girec_map["n_security_spots"],
    c=zones_girec_map["security_ratio"],
    cmap="RdYlGn",
    vmin=0, vmax=1,
    alpha=0.8,
    edgecolors="white",
    linewidth=0.5,
    s=80,
)

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Ratio sécurité / total\n(0 = tout insécurité, 1 = tout sécurité)")

# Calculer la distance minimale entre chaque point et ses voisins
coords = zones_girec_map[["n_insecurity_spots", "n_security_spots"]].values
dist_matrix = cdist(coords, coords)
np.fill_diagonal(dist_matrix, np.inf)  # ignorer la distance à soi-même
min_distances = dist_matrix.min(axis=1)

# N'afficher le label que si le point est suffisamment isolé
threshold = 2  # ← ajuster selon la densité de tes points
for i, (_, row) in enumerate(zones_girec_map.iterrows()):
    if min_distances[i] >= threshold:
        ax.annotate(
            row["NOM"],
            xy=(row["n_insecurity_spots"], row["n_security_spots"]),
            xytext=(3, 5),
            textcoords="offset points",
            fontsize=6,
            alpha=0.8,
        )

ax.set_xlabel("Lieux d'insécurité")
ax.set_ylabel("Lieux de sécurité")
ax.set_title("Distribution des lieux de sécurité et d'insécurité la nuit par sous-secteur GIREC")
plt.tight_layout()
plt.show()

In [ ]:
zones_girec_map

In [ ]:
zones_girec_map.columns

In [ ]:
from scipy import stats as scipy_stats
import plotly.express as px

# Préparer les données (exclure les NaN)
corr_df = zones_girec_map[["n_insecurity_spots", "walk_index_norm", "eclairage"]].dropna()

# Matrice de corrélation
print("Matrice de corrélation (Pearson):")
display(corr_df.corr().round(3))

print("\nMatrice de corrélation (Spearman) — plus robuste pour données non-normales:")
display(corr_df.corr(method="spearman").round(3))

# Tests de significativité pour chaque paire
pairs = [
    ("n_insecurity_spots", "walk_index_norm"),
    ("n_insecurity_spots", "eclairage"),
    ("walk_index_norm",         "eclairage"),
]
print("\nTests de corrélation (Spearman):")
for col_a, col_b in pairs:
    a = corr_df[col_a]
    b = corr_df[col_b]
    r, p = scipy_stats.spearmanr(a, b)
    print(f"  {col_a} × {col_b}: r={r:.3f}, p={p:.4f}", end="  ")
    if p < 0.001:
        print("***")
    elif p < 0.01:
        print("**")
    elif p < 0.05:
        print("*")
    else:
        print("(not significant)")

# Scatter matrix
fig = px.scatter_matrix(
    corr_df,
    dimensions=["n_insecurity_spots", "walk_index_norm", "eclairage"],
    title="Scatter matrix — insécurité, walk_index, éclairage par sous-secteur GIREC",
    labels={
        "n_insecurity_spots": "Spots insécurité",
        "walk_index_norm":         "Walk index",
        "eclairage":          "Éclairage",
    },
    opacity=0.7,
)
fig.update_traces(marker=dict(size=5, color="#E45756"))
fig.update_layout(height=600)
fig.show()

In [ ]:
# Points sans sous-secteur associé
n_missing = PL_spots_security["sous_secteur"].isna().sum()
n_total   = len(PL_spots_security)
print(f"Spots sans sous-secteur: {n_missing} / {n_total} ({n_missing/n_total*100:.1f}%)")
print(f"Spots avec sous-secteur: {n_total - n_missing} / {n_total} ({(n_total-n_missing)/n_total*100:.1f}%)")

In [ ]:
# Afficher les points sans sous-secteur sur la carte
missing_spots = PL_spots_insecurity[PL_spots_insecurity["sous_secteur"].isna()]

fig, ax = plt.subplots(figsize=(12, 10))
zones_girec.plot(ax=ax, color="lightgrey", edgecolor="white", linewidth=0.5)
missing_spots.plot(ax=ax, color="red", markersize=5, label=f"Sans sous-secteur (n={n_missing})")
ax.set_title("Spots d'insécurité sans sous-secteur GIREC")
ax.legend()
ax.set_axis_off()
plt.show()

### Carte sécurité/insécurité par carreau 200

In [ ]:
print(agglo_carreau.columns.tolist())

In [ ]:
# --- Comptages spots par carreau 200m ---

# Spatial join spots → carreaux
spots_insecurity_carreau = gpd.sjoin(
    PL_spots_insecurity, agglo_carreau, how="left", predicate="within"
)
spots_security_carreau = gpd.sjoin(
    PL_spots_security, agglo_carreau, how="left", predicate="within"
)

# Comptages par carreau
id_col = "GRID_ID" 

counts_insecurity_carreau = (
    spots_insecurity_carreau
    .groupby(id_col)
    .size()
    .reset_index(name="n_insecurity_spots")
)

counts_security_carreau = (
    spots_security_carreau
    .groupby(id_col)
    .size()
    .reset_index(name="n_security_spots")
)

# Merge sur la couche carreaux
carreau_map = agglo_carreau.merge(
    counts_insecurity_carreau, on=id_col, how="left"
).merge(
    counts_security_carreau, on=id_col, how="left"
)

carreau_map["n_insecurity_spots"] = carreau_map["n_insecurity_spots"].fillna(0).astype(int)
carreau_map["n_security_spots"]   = carreau_map["n_security_spots"].fillna(0).astype(int)
carreau_map["n_total_spots"]      = carreau_map["n_insecurity_spots"] + carreau_map["n_security_spots"]

print(f"Carreaux avec au moins 1 spot insécurité: {(carreau_map['n_insecurity_spots'] > 0).sum()}")
print(f"Carreaux avec au moins 1 spot sécurité:   {(carreau_map['n_security_spots'] > 0).sum()}")
print(f"\nTop 10 carreaux — insécurité:")
display(carreau_map[[id_col, "n_insecurity_spots", "n_security_spots", "n_total_spots"]]
        .sort_values("n_insecurity_spots", ascending=False)
        .head(10))

In [ ]:
len(carreau_map)

In [ ]:
plot_spots_map(carreau_map, "n_insecurity_spots", "Reds",
               "Spots d'insécurité par carreau 200m", "Spots insécurité")

plot_spots_map(carreau_map, "n_security_spots", "Greens",
               "Spots de sécurité par carreau 200m", "Spots sécurité")

# STAT ON ALL RESPONDANT

## P1_Q12a — Temps de qualité lors des déplacements quotidiens

In [ ]:
import plotly.graph_objects as go

mode_labels = {
    "P1_Q12a_1": "Voiture",
    "P1_Q12a_2": "Train",
    "P1_Q12a_3": "TP (sans train)",
    "P1_Q12a_4": "Vélo",
    "P1_Q12a_5": "Moto / scooter",
    "P1_Q12a_6": "Marche à pied",
    "P1_Q12a_7": "Bateau",
    "P1_Q12a_8": "Trottinette",
}

score_labels = {
    1: "1 — Pas du tout",
    2: "2 — Plutôt non",
    3: "3 — Ni oui, ni non",
    4: "4 — Plutôt oui",
    5: "5 — Tout à fait",
}

# Build proportions per mode and per score
rows = []
for col, mode_name in mode_labels.items():
    series = pd.to_numeric(PL_respondant_rythme[col], errors="coerce").dropna()
    total = len(series)
    for score, label in score_labels.items():
        count = (series == score).sum()
        pct = count / total * 100 if total > 0 else 0
        rows.append({"mode": mode_name, "score": score, "score_label": label, "pct": round(pct, 1), "n": total})

q12a_df = pd.DataFrame(rows)

# Summary table
print("Total respondant to rythm survey :", len(PL_respondant_rythme))
print("N répondants par mode (excl. NA):")
print(q12a_df.groupby("mode")["n"].first().to_string())
print()

summary = q12a_df.pivot(index="mode", columns="score_label", values="pct")
display(summary)

In [ ]:
color_map = {
    "1 — Pas du tout":    "#d73027",
    "2 — Plutôt non":     "#fc8d59",
    "3 — Ni oui, ni non": "#d9d9d9",
    "4 — Plutôt oui":     "#91bfdb",
    "5 — Tout à fait":    "#4575b4",
}

fig = go.Figure()

# Order: 1 at bottom → 5 at top
for score in [1, 2, 3, 4, 5]:
    label = score_labels[score]
    subset = q12a_df[q12a_df["score"] == score]
    fig.add_trace(go.Bar(
        name=label,
        x=subset["mode"],
        y=subset["pct"],
        text=subset["pct"].apply(lambda v: f"{v:.0f}%" if v >= 5 else ""),  # hide tiny labels
        textposition="inside",
        insidetextanchor="middle",
        marker_color=color_map[label],
    ))

fig.update_layout(
    barmode="stack",
    title="P1_Q12a — Possibilité d'avoir du temps de qualité lors des déplacements quotidiens",
    xaxis_title="Mode de transport",
    yaxis=dict(title="Proportion (%)", range=[0, 100], ticksuffix="%"),
    legend_title="Score",
    legend=dict(traceorder="normal"),
    height=520,
    xaxis_tickangle=-20,
)
fig.show()

In [ ]:
# % of positive responses (score 4 + 5) per mode, sorted descending
positive = (
    q12a_df[q12a_df["score"].isin([4, 5])]
    .groupby("mode")["pct"]
    .sum()
    .reset_index()
    .rename(columns={"pct": "pct_positive"})
    .sort_values("pct_positive", ascending=True)  # ascending for horizontal bar (best at top)
)

fig = px.bar(
    positive,
    x="pct_positive",
    y="mode",
    orientation="h",
    text=positive["pct_positive"].apply(lambda v: f"{v:.1f}%"),
    title="P1_Q12a — Classement des modes selon le temps de qualité (scores 4 + 5 combinés)",
    labels={"pct_positive": "% répondants (score 4 ou 5)", "mode": "Mode de transport"},
    color="pct_positive",
    color_continuous_scale=["#fc8d59", "#ffffbf", "#4575b4"],
    range_color=[0, 100],
)
fig.update_traces(textposition="outside")
fig.update_layout(
    xaxis=dict(range=[0, 108], ticksuffix="%"),
    coloraxis_showscale=False,
    height=420,
)
fig.show()

## P1_Q13b — Ce que donnerait du temps de qualité lors des déplacements
*(Filtré sur P1_Q13c_6 = 1 — ceux ayant sélectionné la marche à pied)*

In [ ]:
q13b_labels = {
    "P1_Q13b_1": "Se reposer",
    "P1_Q13b_2": "Faire des activités",
    "P1_Q13b_3": "Apprécier pleinement le trajet",
    "P1_Q13b_4": "Gagner du temps",
    "P1_Q13b_5": "Réduire stress / se détendre",
    "P1_Q13b_6": "Rencontrer / parler à d'autres",
}

# Filter: only respondents who selected P1_Q13c_6 = 1
df_q13b = PL_respondant_rythme[pd.to_numeric(PL_respondant_rythme["P1_Q13c_6"], errors="coerce") == 1].copy()
print(f"Respondants avec P1_Q13c_6 = 1 : {len(df_q13b)} / {len(PL_respondant_rythme)} ({len(df_q13b) / len(PL_respondant_rythme) * 100:.2f} %)")

# Build proportions per benefit and per score
rows_q13b = []
for col, label in q13b_labels.items():
    series = pd.to_numeric(df_q13b[col], errors="coerce").dropna()
    total = len(series)
    for score, score_label in score_labels.items():
        count = (series == score).sum()
        pct = count / total * 100 if total > 0 else 0
        rows_q13b.append({"benefit": label, "score": score, "score_label": score_label, "pct": round(pct, 1), "n": total})

q13b_df = pd.DataFrame(rows_q13b)

print("\nN répondants par item (excl. NA):")
print(q13b_df.groupby("benefit")["n"].first().to_string())
print()
display(q13b_df.pivot(index="benefit", columns="score_label", values="pct"))

In [ ]:
# 100% stacked bar chart
fig = go.Figure()

for score in [1, 2, 3, 4, 5]:
    label = score_labels[score]
    subset = q13b_df[q13b_df["score"] == score]
    fig.add_trace(go.Bar(
        name=label,
        x=subset["benefit"],
        y=subset["pct"],
        text=subset["pct"].apply(lambda v: f"{v:.0f}%" if v >= 5 else ""),
        textposition="inside",
        insidetextanchor="middle",
        marker_color=color_map[label],
    ))

fig.update_layout(
    barmode="stack",
    title="P1_Q13b — Ce que donnerait du temps de qualité lors des déplacements<br>(filtré P1_Q13c_6 = 1 --> Mode référent = marche à pied)",
    xaxis_title="Bénéfice potentiel",
    yaxis=dict(title="Proportion (%)", range=[0, 100], ticksuffix="%"),
    legend_title="Score",
    legend=dict(traceorder="normal"),
    height=520,
    xaxis_tickangle=-20,
)
fig.show()

In [ ]:
# Ranking chart: % scores 4 + 5
positive_q13b = (
    q13b_df[q13b_df["score"].isin([4, 5])]
    .groupby("benefit")["pct"]
    .sum()
    .reset_index()
    .rename(columns={"pct": "pct_positive"})
    .sort_values("pct_positive", ascending=True)
)

fig = px.bar(
    positive_q13b,
    x="pct_positive",
    y="benefit",
    orientation="h",
    text=positive_q13b["pct_positive"].apply(lambda v: f"{v:.1f}%"),
    title="P1_Q13b — Classement des bénéfices attendus (scores 4 + 5 combinés)<br>(filtré P1_Q13c_6 = 1 --> Mode référent = marche à pied)",
    labels={"pct_positive": "% répondants (score 4 ou 5)", "benefit": "Bénéfice potentiel"},
    color="pct_positive",
    color_continuous_scale=["#fc8d59", "#ffffbf", "#4575b4"],
    range_color=[0, 100],
)
fig.update_traces(textposition="outside")
fig.update_layout(
    xaxis=dict(range=[0, 108], ticksuffix="%"),
    coloraxis_showscale=False,
    height=420,
)
fig.show()

## P2_Q6 — Sentiment de sécurité de jour, par mode de transport

In [ ]:
q6_mode_labels = {
    "P2_Q6_1": "Voiture",
    "P2_Q6_2": "Train",
    "P2_Q6_3": "TP (sans train)",
    "P2_Q6_4": "Vélo",
    "P2_Q6_5": "Moto / scooter",
    "P2_Q6_6": "Marche à pied",
    "P2_Q6_7": "Bateau",
    "P2_Q6_8": "Trottinette",
    "P2_Q6_9": "Taxi",
}

q6_score_labels = {
    1: "1 — Pas du tout en sécurité",
    2: "2 — Plutôt pas en sécurité",
    3: "3 — Ni en sécurité, ni pas",
    4: "4 — Plutôt en sécurité",
    5: "5 — Tout à fait en sécurité",
}

q6_color_map = {
    "1 — Pas du tout en sécurité":  "#d73027",
    "2 — Plutôt pas en sécurité":   "#fc8d59",
    "3 — Ni en sécurité, ni pas":   "#d9d9d9",
    "4 — Plutôt en sécurité":       "#91bfdb",
    "5 — Tout à fait en sécurité":  "#4575b4",
}

# Build proportions per mode and per score
rows_q6 = []
for col, mode_name in q6_mode_labels.items():
    series = pd.to_numeric(PL_respondant_rythme[col], errors="coerce").dropna()
    total = len(series)
    for score, label in q6_score_labels.items():
        count = (series == score).sum()
        pct = count / total * 100 if total > 0 else 0
        rows_q6.append({"mode": mode_name, "score": score, "score_label": label, "pct": round(pct, 1), "n": total})

q6_df = pd.DataFrame(rows_q6)

print("Total respondant to rythm survey :", len(PL_respondant_rythme))
print("N répondants par mode (excl. NA):")
print(q6_df.groupby("mode")["n"].first().to_string())
print()
display(q6_df.pivot(index="mode", columns="score_label", values="pct"))

In [ ]:
# 100% stacked bar chart
fig = go.Figure()

for score in [1, 2, 3, 4, 5]:
    label = q6_score_labels[score]
    subset = q6_df[q6_df["score"] == score]
    fig.add_trace(go.Bar(
        name=label,
        x=subset["mode"],
        y=subset["pct"],
        text=subset["pct"].apply(lambda v: f"{v:.0f}%" if v >= 5 else ""),
        textposition="inside",
        insidetextanchor="middle",
        marker_color=q6_color_map[label],
    ))

fig.update_layout(
    barmode="stack",
    title="P2_Q6 — Sentiment de sécurité de jour, par mode de transport",
    xaxis_title="Mode de transport",
    yaxis=dict(title="Proportion (%)", range=[0, 100], ticksuffix="%"),
    legend_title="Score",
    legend=dict(traceorder="normal"),
    height=520,
    xaxis_tickangle=-20,
)
fig.show()

# Ranking chart: % scores 4 + 5
positive_q6 = (
    q6_df[q6_df["score"].isin([4, 5])]
    .groupby("mode")["pct"]
    .sum()
    .reset_index()
    .rename(columns={"pct": "pct_positive"})
    .sort_values("pct_positive", ascending=True)
)

fig2 = px.bar(
    positive_q6,
    x="pct_positive",
    y="mode",
    orientation="h",
    text=positive_q6["pct_positive"].apply(lambda v: f"{v:.1f}%"),
    title="P2_Q6 — Classement des modes selon le sentiment de sécurité de jour (scores 4 + 5)",
    labels={"pct_positive": "% répondants (score 4 ou 5)", "mode": "Mode de transport"},
    color="pct_positive",
    color_continuous_scale=["#fc8d59", "#ffffbf", "#4575b4"],
    range_color=[0, 100],
)
fig2.update_traces(textposition="outside")
fig2.update_layout(
    xaxis=dict(range=[0, 108], ticksuffix="%"),
    coloraxis_showscale=False,
    height=420,
)
fig2.show()

# Ranking chart: % scores 1 + 2
negative_q6 = (
    q6_df[q6_df["score"].isin([1, 2])]
    .groupby("mode")["pct"]
    .sum()
    .reset_index()
    .rename(columns={"pct": "pct_negative"})
    .sort_values("pct_negative", ascending=True)
)

fig3 = px.bar(
    negative_q6,
    x="pct_negative",
    y="mode",
    orientation="h",
    text=negative_q6["pct_negative"].apply(lambda v: f"{v:.1f}%"),
    title="P2_Q6 — Classement des modes selon le sentiment d'insécurité de jour (scores 1 + 2)",
    labels={"pct_negative": "% répondants (score 1 ou 2)", "mode": "Mode de transport"},
    color="pct_negative",
    color_continuous_scale=["#4575b4",  "#ffffbf", "#fc8d59"],
    range_color=[0, 100],
)
fig3.update_traces(textposition="outside")
fig3.update_layout(
    xaxis=dict(range=[0, 108], ticksuffix="%"),
    coloraxis_showscale=False,
    height=420,
)
fig3.show()

## P2_Q7 — Sentiment de sécurité de nuit, par mode de transport

In [ ]:
q7_mode_labels = {
    "P2_Q7_1": "Voiture",
    "P2_Q7_2": "Train",
    "P2_Q7_3": "TP (sans train)",
    "P2_Q7_4": "Vélo",
    "P2_Q7_5": "Moto / scooter",
    "P2_Q7_6": "Marche à pied",
    "P2_Q7_7": "Bateau",
    "P2_Q7_8": "Trottinette",
    "P2_Q7_9": "Taxi",
}

# Same score labels and color map as Q6
q7_score_labels = q6_score_labels
q7_color_map = q6_color_map

# Build proportions per mode and per score
rows_q7 = []
for col, mode_name in q7_mode_labels.items():
    series = pd.to_numeric(PL_respondant_rythme[col], errors="coerce").dropna()
    total = len(series)
    for score, label in q7_score_labels.items():
        count = (series == score).sum()
        pct = count / total * 100 if total > 0 else 0
        rows_q7.append({"mode": mode_name, "score": score, "score_label": label, "pct": round(pct, 1), "n": total})

q7_df = pd.DataFrame(rows_q7)

print("Total respondant to rythm survey :", len(PL_respondant_rythme))
print("N répondants par mode (excl. NA):")
print(q7_df.groupby("mode")["n"].first().to_string())
print()
display(q7_df.pivot(index="mode", columns="score_label", values="pct"))

In [ ]:
# 100% stacked bar chart
fig = go.Figure()

for score in [1, 2, 3, 4, 5]:
    label = q7_score_labels[score]
    subset = q7_df[q7_df["score"] == score]
    fig.add_trace(go.Bar(
        name=label,
        x=subset["mode"],
        y=subset["pct"],
        text=subset["pct"].apply(lambda v: f"{v:.0f}%" if v >= 5 else ""),
        textposition="inside",
        insidetextanchor="middle",
        marker_color=q7_color_map[label],
    ))

fig.update_layout(
    barmode="stack",
    title="P2_Q7 — Sentiment de sécurité de nuit, par mode de transport",
    xaxis_title="Mode de transport",
    yaxis=dict(title="Proportion (%)", range=[0, 100], ticksuffix="%"),
    legend_title="Score",
    legend=dict(traceorder="normal"),
    height=520,
    xaxis_tickangle=-20,
)
fig.show()

# Ranking chart: % scores 4 + 5
positive_q7 = (
    q7_df[q7_df["score"].isin([4, 5])]
    .groupby("mode")["pct"]
    .sum()
    .reset_index()
    .rename(columns={"pct": "pct_positive"})
    .sort_values("pct_positive", ascending=True)
)

fig2 = px.bar(
    positive_q7,
    x="pct_positive",
    y="mode",
    orientation="h",
    text=positive_q7["pct_positive"].apply(lambda v: f"{v:.1f}%"),
    title="P2_Q7 — Classement des modes selon le sentiment de sécurité de nuit (scores 4 + 5)",
    labels={"pct_positive": "% répondants (score 4 ou 5)", "mode": "Mode de transport"},
    color="pct_positive",
    color_continuous_scale=["#fc8d59", "#ffffbf", "#4575b4"],
    range_color=[0, 100],
)
fig2.update_traces(textposition="outside")
fig2.update_layout(
    xaxis=dict(range=[0, 108], ticksuffix="%"),
    coloraxis_showscale=False,
    height=420,
)
fig2.show()

# Ranking chart: % scores 1 + 2
negative_q7 = (
    q7_df[q7_df["score"].isin([1, 2])]
    .groupby("mode")["pct"]
    .sum()
    .reset_index()
    .rename(columns={"pct": "pct_negative"})
    .sort_values("pct_negative", ascending=True)
)

fig3 = px.bar(
    negative_q7,
    x="pct_negative",
    y="mode",
    orientation="h",
    text=negative_q7["pct_negative"].apply(lambda v: f"{v:.1f}%"),
    title="P2_Q7 — Classement des modes selon le sentiment d'insécurité de nuit (scores 1 + 2)",
    labels={"pct_negative": "% répondants (score 1 ou 2)", "mode": "Mode de transport"},
    color="pct_negative",
    color_continuous_scale=["#4575b4", "#ffffbf", "#fc8d59"],
    range_color=[0, 100],
)
fig3.update_traces(textposition="outside")
fig3.update_layout(
    xaxis=dict(range=[0, 108], ticksuffix="%"),
    coloraxis_showscale=False,
    height=420,
)
fig3.show()

## P2_Q8 — Tendance à choisir un mode différent la nuit

In [ ]:
q8_answer_labels = {1: "Oui", 2: "Plutot oui", 3: "Ni oui ni non", 4: "Plutot non", 5: "Non", 6: "Sans avis"}
q8_answer_colors = {"Oui": "#d73027", "Plutot oui": "#fc8d59", "Ni oui ni non": "#d9d9d9", "Plutot non": "#91bfdb", "Non": "#4575b4", "Sans avis": "#969696"}
q8_order = ["Oui", "Plutot oui", "Ni oui ni non", "Plutot non", "Non", "Sans avis"]

df_q8 = PL_respondant_rythme[["P2_Q8", "P0_genre"]].copy()
df_q8["score"]     = pd.to_numeric(df_q8["P2_Q8"],    errors="coerce")
df_q8["genre_int"] = pd.to_numeric(df_q8["P0_genre"], errors="coerce")
df_q8["gender"]    = df_q8["genre_int"].map({1: "Man", 2: "Woman", 3: "Other", 4: "Other", 5: "Other"})
df_q8 = df_q8.dropna(subset=["score"])
df_q8["answer"] = df_q8["score"].map(q8_answer_labels)
total_q8 = len(df_q8)

# Overall
overall_q8 = df_q8["answer"].value_counts().reindex(q8_order, fill_value=0).reset_index()
overall_q8.columns = ["answer", "count"]
overall_q8["pct"] = (overall_q8["count"] / total_q8 * 100).round(1)

print(f"N respondants: {total_q8} / {len(PL_respondant_rythme)}")
display(overall_q8.set_index("answer"))
pos = overall_q8[overall_q8["answer"].isin(["Oui", "Plutot oui"])]["pct"].sum()
neg = overall_q8[overall_q8["answer"].isin(["Plutot non", "Non"])]["pct"].sum()
print(f"\nChange mode at night (1+2): {pos:.1f}%")
print(f"Keep same mode       (4+5): {neg:.1f}%")

# Gender sample sizes
n_by_gender = df_q8["gender"].value_counts().rename("N")
print(f"\nN per gender: {n_by_gender.to_dict()}")
print("Note: 'Other' excluded from gender breakdown (n too small for reliable %)")

# By gender — Man and Woman only
gender_q8 = (
    df_q8[df_q8["gender"].isin(["Man", "Woman"])]
    .groupby(["gender", "answer"]).size().reset_index(name="count")
)
gender_q8["total"] = gender_q8.groupby("gender")["count"].transform("sum")
gender_q8["pct"]   = (gender_q8["count"] / gender_q8["total"] * 100).round(1)

print("\nBy gender (count):")
display(gender_q8.pivot(index="answer", columns="gender", values="count").reindex(q8_order))
print("\nBy gender (%):")
display(gender_q8.pivot(index="answer", columns="gender", values="pct").reindex(q8_order))

In [ ]:
gender_colors = {"Man": "#4575b4", "Woman": "#d73027"}

fig = px.bar(
    overall_q8,
    x="answer", y="pct",
    color="answer",
    text=overall_q8["pct"].astype(str) + "%",
    title="P2_Q8 - Tendency to change transport mode at night (overall)",
    labels={"answer": "Answer", "pct": "% respondants"},
    color_discrete_map=q8_answer_colors,
    category_orders={"answer": q8_order},
)
fig.update_traces(textposition="outside", showlegend=False)
fig.update_layout(yaxis=dict(range=[0, 55], ticksuffix="%"), height=450)
fig.show()

fig2 = px.bar(
    gender_q8,
    x="answer", y="pct",
    color="gender",
    barmode="group",
    text=gender_q8["pct"].astype(str) + "%",
    title="P2_Q8 - Tendency to change transport mode at night, by gender (Man / Woman)",
    labels={"answer": "Answer", "pct": "% within gender", "gender": "Gender"},
    color_discrete_map=gender_colors,
    category_orders={"answer": q8_order, "gender": ["Man", "Woman"]},
)
fig2.update_traces(textposition="outside")
fig2.update_layout(yaxis=dict(range=[0, 55], ticksuffix="%"), height=480, legend_title="Gender")
fig2.show()

## P2_Q9 — Modes de transport preferés la nuit (max 2 choix)

In [ ]:
q9_mode_labels = {
    "P2_Q9_1": "Voiture",
    "P2_Q9_2": "Train",
    "P2_Q9_3": "TP (sans train)",
    "P2_Q9_4": "Velo",
    "P2_Q9_5": "Moto / scooter",
    "P2_Q9_6": "Marche a pied",
    "P2_Q9_7": "Bateau",
    "P2_Q9_8": "Trottinette",
    "P2_Q9_9": "Taxi",
}

# Add gender to working dataframe
df_q9 = PL_respondant_rythme.copy()
df_q9["genre_int"] = pd.to_numeric(df_q9["P0_genre"], errors="coerce")
df_q9["gender"]    = df_q9["genre_int"].map({1: "Man", 2: "Woman", 3: "Other", 4: "Other", 5: "Other"})
n_total   = len(df_q9)


# N who actually answered at least one Q10 column
answered_q9 = df_q9[[*q9_mode_labels]].apply(pd.to_numeric, errors="coerce").notna().any(axis=1)
n_answered     = answered_q9.sum()
n_man_q9     = (df_q9.loc[answered_q9, "gender"] == "Man").sum()
n_woman_q9   = (df_q9.loc[answered_q9, "gender"] == "Woman").sum()
n_other_q9 = (df_q9.loc[answered_q9, "gender"] == "Other").sum()
print(f"Respondants having answered Q9: {n_answered} / {len(df_q9)}")
print(f"  Man: {n_man_q9}  |  Woman: {n_woman_q9}  |  Other: {n_other_q9} but excluded because too low answer")
print("(N per behavior may vary slightly due to item-level NA)\n")



print(f"Total respondants: {n_answered}  |  Man: {n_man_q9}  |  Woman: {n_woman_q9}  |  Other: {n_other_q9} (excluded from gender breakdown)")

# Overall: % who selected each mode
rows_q9 = []
for col, mode in q9_mode_labels.items():
    series = pd.to_numeric(df_q9[col], errors="coerce")
    n_valid = series.notna().sum() #only nb of respondant that answer the question, skip NA rows
    n_selected = (series == 1).sum()
    pct = round(n_selected / n_valid * 100, 1)
    rows_q9.append({"mode": mode, "count": n_selected, "n_valid": n_valid, "pct": pct})

q9_overall = pd.DataFrame(rows_q9).sort_values("pct", ascending=False)
print("\nOverall — % selecting each mode as preferred at night:")
display(q9_overall.set_index("mode"))

# By gender: % who selected each mode
rows_q9g = []
for gender in ["Man", "Woman"]:
    subset = df_q9[df_q9["gender"] == gender]
    for col, mode in q9_mode_labels.items():
        series = pd.to_numeric(subset[col], errors="coerce")
        n_valid = series.notna().sum()
        n_selected = (series == 1).sum()
        pct = round(n_selected / n_valid * 100, 1)
        rows_q9g.append({"gender": gender, "mode": mode, "count": n_selected, "pct": pct})

q9_gender = pd.DataFrame(rows_q9g)

print("\nBy gender (%):")
display(q9_gender.pivot(index="mode", columns="gender", values="pct").sort_values("Man", ascending=False))

print("\nBy gender (count):")
display(q9_gender.pivot(index="mode", columns="gender", values="count").sort_values("Man", ascending=False))

In [ ]:
# Overall ranking
fig = px.bar(
    q9_overall.sort_values("pct", ascending=True),
    x="pct", y="mode", orientation="h",
    text=q9_overall.sort_values("pct", ascending=True)["pct"].astype(str) + "%",
    title="P2_Q9 — Modes de transport preferes la nuit (% ayant selectionne ce mode)",
    labels={"pct": "% respondants", "mode": "Mode"},
    color="pct",
    color_continuous_scale=["#fc8d59", "#ffffbf", "#4575b4"],
    range_color=[0, 100],
)
fig.update_traces(textposition="outside")
fig.update_layout(xaxis=dict(range=[0, 108], ticksuffix="%"), coloraxis_showscale=False, height=440)
fig.show()

# By gender — grouped bar
fig2 = px.bar(
    q9_gender,
    x="mode", y="pct", color="gender",
    barmode="group",
    text=q9_gender["pct"].astype(str) + "%",
    title="P2_Q9 — Modes preferes la nuit par genre (% ayant selectionne ce mode)",
    labels={"mode": "Mode", "pct": "% dans chaque genre", "gender": "Genre"},
    color_discrete_map={"Man": "#4575b4", "Woman": "#d73027"},
    category_orders={"gender": ["Man", "Woman"]},
)
fig2.update_traces(textposition="outside")
fig2.update_layout(
    yaxis=dict(range=[0, 105], ticksuffix="%"),
    xaxis_tickangle=-20, height=500, legend_title="Genre",
)
fig2.show()

## P2_Q10 — Raisons de changer de mode la nuit (max 2 réponses)

In [ ]:
q10_reason_labels = {
    "P2_Q10_1": "Securite routiere",
    "P2_Q10_2": "Flexibilite",
    "P2_Q10_3": "Mode moins contraignant",
    "P2_Q10_4": "Deplacement a rythme tranquille",
    "P2_Q10_5": "Risque mauvaise rencontre (infra transport)",
    "P2_Q10_6": "Risque mauvaise rencontre (espace public)",
    "P2_Q10_7": "Raison economique",
    "P2_Q10_8": "Autre",
}

df_q10 = PL_respondant_rythme.copy()
df_q10["genre_int"] = pd.to_numeric(df_q10["P0_genre"], errors="coerce")
df_q10["gender"]    = df_q10["genre_int"].map({1: "Man", 2: "Woman", 3: "Other", 4: "Other", 5: "Other"})
"""



n_man_q10   = (df_q10["gender"] == "Man").sum()
n_woman_q10 = (df_q10["gender"] == "Woman").sum()
print(f"Total respondants: {len(df_q10)}  |  Man: {n_man_q10}  |  Woman: {n_woman_q10}  |  Other: {(df_q10['gender']=='Other').sum()} (excluded)")
"""


# N who actually answered at least one Q10 column
answered_q10 = df_q10[[*q10_reason_labels]].apply(pd.to_numeric, errors="coerce").notna().any(axis=1)
n_answered     = answered_q10.sum()
n_man_q10     = (df_q10.loc[answered_q10, "gender"] == "Man").sum()
n_woman_q10   = (df_q10.loc[answered_q10, "gender"] == "Woman").sum()
n_other_q10 = (df_q10.loc[answered_q10, "gender"] == "Other").sum()
print(f"Respondants having answered Q10: {n_answered} / {len(df_q10)}")
print(f"  Man: {n_man_q10}  |  Woman: {n_woman_q10}  |  Other: {n_other_q10} but excluded because too low answer")
print("(N per behavior may vary slightly due to item-level NA)\n")


# Overall
rows_q10 = []
for col, reason in q10_reason_labels.items():
    series = pd.to_numeric(df_q10[col], errors="coerce")
    n_valid    = series.notna().sum()
    n_selected = (series == 1).sum()
    pct = round(n_selected / n_valid * 100, 1) if n_valid > 0 else 0
    rows_q10.append({"reason": reason, "count": n_selected, "n_valid": n_valid, "pct": pct})

q10_overall = pd.DataFrame(rows_q10).sort_values("pct", ascending=False)
print("\nOverall — % selecting each reason:")
display(q10_overall.set_index("reason"))

# By gender (Man / Woman only)
rows_q10g = []
for gender in ["Man", "Woman"]:
    subset = df_q10[df_q10["gender"] == gender]
    for col, reason in q10_reason_labels.items():
        series = pd.to_numeric(subset[col], errors="coerce")
        n_valid    = series.notna().sum()
        n_selected = (series == 1).sum()
        pct = round(n_selected / n_valid * 100, 1) if n_valid > 0 else 0
        rows_q10g.append({"gender": gender, "reason": reason, "count": n_selected, "pct": pct})

q10_gender = pd.DataFrame(rows_q10g)

print("\nBy gender (%):")
display(q10_gender.pivot(index="reason", columns="gender", values="pct").sort_values("Man", ascending=False))
print("\nBy gender (count):")
display(q10_gender.pivot(index="reason", columns="gender", values="count").sort_values("Man", ascending=False))

In [ ]:
fig = px.bar(
    q10_overall.sort_values("pct", ascending=True),
    x="pct", y="reason", orientation="h",
    text=q10_overall.sort_values("pct", ascending=True)["pct"].astype(str) + "%",
    title="P2_Q10 — Raisons de changer de mode la nuit (tous les respondants)",
    labels={"pct": "% respondants", "reason": "Raison"},
    color="pct",
    color_continuous_scale=["#fc8d59", "#ffffbf", "#4575b4"],
    range_color=[0, 100],
)
fig.update_traces(textposition="outside")
fig.update_layout(xaxis=dict(range=[0, 108], ticksuffix="%"), coloraxis_showscale=False, height=460)
fig.show()

fig2 = px.bar(
    q10_gender,
    x="reason", y="pct", color="gender",
    barmode="group",
    text=q10_gender["pct"].astype(str) + "%",
    title="P2_Q10 — Raisons de changer de mode la nuit par genre",
    labels={"reason": "Raison", "pct": "% dans chaque genre", "gender": "Genre"},
    color_discrete_map={"Man": "#4575b4", "Woman": "#d73027"},
    category_orders={"gender": ["Man", "Woman"]},
)
fig2.update_traces(textposition="outside")
fig2.update_layout(
    yaxis=dict(range=[0, 105], ticksuffix="%"),
    xaxis_tickangle=-20, height=520, legend_title="Genre",
)
fig2.show()

## P2_Q11a — Comportements de marche la nuit

In [ ]:
q11a_labels = {
    "P2_Q11a_1": "Evite rentrer apres certaine heure",
    "P2_Q11a_2": "Evite espaces ouverts (parking, zone commerciale)",
    "P2_Q11a_3": "Evite rues animation nocturne (bars)",
    "P2_Q11a_4": "Evite routes vehicules trop rapides",
    "P2_Q11a_5": "Evite sortir a pied, prefere autre mode",
}

df_q11a = PL_respondant_rythme.copy()
df_q11a["genre_int"] = pd.to_numeric(df_q11a["P0_genre"], errors="coerce")
df_q11a["gender"]    = df_q11a["genre_int"].map({1: "Man", 2: "Woman", 3: "Other", 4: "Other", 5: "Other"})

# Exclude "Other" for all computations
df_q11a = df_q11a[df_q11a["gender"].isin(["Man", "Woman"])]
print(f"Respondants (Man + Woman only): {len(df_q11a)}  —  Man: {(df_q11a['gender']=='Man').sum()}, Woman: {(df_q11a['gender']=='Woman').sum()}")

# Overall proportions
rows_q11a = []
for col, behavior in q11a_labels.items():
    series = pd.to_numeric(df_q11a[col], errors="coerce").dropna()
    total = len(series)
    for score, label in score_labels.items():
        count = (series == score).sum()
        rows_q11a.append({"behavior": behavior, "score": score, "score_label": label, "pct": round(count/total*100, 1), "n": total})

q11a_df = pd.DataFrame(rows_q11a)

print("\nN respondants per behavior (excl. NA, excl. Other):")
print(q11a_df.groupby("behavior")["n"].first().to_string())
print()
display(q11a_df.pivot(index="behavior", columns="score_label", values="pct"))

# By gender — N per behavior per gender
print("\nN per behavior per gender:")
for col, behavior in q11a_labels.items():
    n_m = pd.to_numeric(df_q11a.loc[df_q11a["gender"]=="Man",   col], errors="coerce").notna().sum()
    n_w = pd.to_numeric(df_q11a.loc[df_q11a["gender"]=="Woman", col], errors="coerce").notna().sum()
    print(f"  {behavior}: Man={n_m}, Woman={n_w}")

# By gender proportions
rows_q11ag = []
for gender in ["Man", "Woman"]:
    subset = df_q11a[df_q11a["gender"] == gender]
    for col, behavior in q11a_labels.items():
        series = pd.to_numeric(subset[col], errors="coerce").dropna()
        total = len(series)
        for score, label in score_labels.items():
            count = (series == score).sum()
            rows_q11ag.append({"gender": gender, "behavior": behavior, "score": score, "score_label": label, "pct": round(count/total*100, 1), "n": total})

q11a_gender_df = pd.DataFrame(rows_q11ag)

pos_q11a_gender = (
    q11a_gender_df[q11a_gender_df["score"].isin([4, 5])]
    .groupby(["gender", "behavior"])["pct"].sum().round(1)
    .reset_index().rename(columns={"pct": "pct_positive"})
)
print("\n% positive (scores 4+5) by behavior and gender:")
display(pos_q11a_gender.pivot(index="behavior", columns="gender", values="pct_positive"))

In [ ]:
# 100% stacked bar — overall
fig = go.Figure()
for score in [1, 2, 3, 4, 5]:
    label = score_labels[score]
    subset = q11a_df[q11a_df["score"] == score]
    fig.add_trace(go.Bar(
        name=label, x=subset["behavior"], y=subset["pct"],
        text=subset["pct"].apply(lambda v: f"{v:.0f}%" if v >= 5 else ""),
        textposition="inside", insidetextanchor="middle",
        marker_color=color_map[label],
    ))
fig.update_layout(
    barmode="stack",
    title="P2_Q11a — Comportements de marche la nuit",
    xaxis_title="Comportement", yaxis=dict(title="Proportion (%)", range=[0, 100], ticksuffix="%"),
    legend_title="Score", legend=dict(traceorder="normal"), height=520, xaxis_tickangle=-20,
)
fig.show()

# Positive ranking (scores 4+5) — overall
pos_q11a = (
    q11a_df[q11a_df["score"].isin([4, 5])].groupby("behavior")["pct"].sum()
    .reset_index().rename(columns={"pct": "pct_positive"})
    .sort_values("pct_positive", ascending=True)
)
fig2 = px.bar(
    pos_q11a, x="pct_positive", y="behavior", orientation="h",
    text=pos_q11a["pct_positive"].apply(lambda v: f"{v:.1f}%"),
    title="P2_Q11a — Comportements les plus frequents la nuit (scores 4+5)",
    labels={"pct_positive": "% (score 4 ou 5)", "behavior": "Comportement"},
    color="pct_positive", color_continuous_scale=["#fc8d59", "#ffffbf", "#d73027"], range_color=[0, 100],
)
fig2.update_traces(textposition="outside")
fig2.update_layout(xaxis=dict(range=[0, 108], ticksuffix="%"), coloraxis_showscale=False, height=400)
fig2.show()

# Gender comparison — % positive (4+5) per behavior
fig3 = px.bar(
    pos_q11a_gender,
    x="behavior", y="pct_positive", color="gender",
    barmode="group",
    text=pos_q11a_gender["pct_positive"].astype(str) + "%",
    title="P2_Q11a — Comportements la nuit par genre (scores 4+5)",
    labels={"behavior": "Comportement", "pct_positive": "% (score 4 ou 5)", "gender": "Genre"},
    color_discrete_map={"Man": "#4575b4", "Woman": "#d73027"},
    category_orders={"gender": ["Man", "Woman"]},
)
fig3.update_traces(textposition="outside")
fig3.update_layout(
    yaxis=dict(range=[0, 105], ticksuffix="%"),
    xaxis_tickangle=-20, height=500, legend_title="Genre",
)
fig3.show()

## P2_Q11b — Comportements d'evitement la nuit pour raisons de securite

In [ ]:
q11b_labels = {
    "P2_Q11b_1": "Evite endroits traverses de jour",
    "P2_Q11b_2": "Evite endroits peu eclaires",
    "P2_Q11b_3": "Evite endroits etroits sans echappatoire (ruelle, escalier)",
    "P2_Q11b_4": "Evite endroits entoures de verdure (parc, foret, lac)",
    "P2_Q11b_5": "Evite endroits mal frequentes (groupe/personne menacante)",
    "P2_Q11b_6": "Evite endroits peu frequentes",
    "P2_Q11b_7": "Evite endroits trop eclaires / trop visible",
}

df_q11b = PL_respondant_rythme.copy()
df_q11b["genre_int"] = pd.to_numeric(df_q11b["P0_genre"], errors="coerce")
df_q11b["gender"]    = df_q11b["genre_int"].map({1: "Man", 2: "Woman", 3: "Other", 4: "Other", 5: "Other"})
df_q11b = df_q11b[df_q11b["gender"].isin(["Man", "Woman"])]

print(f"Respondants (Man + Woman only): {len(df_q11b)}  —  Man: {(df_q11b['gender']=='Man').sum()}, Woman: {(df_q11b['gender']=='Woman').sum()}")

# Overall proportions
rows_q11b = []
for col, behavior in q11b_labels.items():
    series = pd.to_numeric(df_q11b[col], errors="coerce").dropna()
    total = len(series)
    for score, label in score_labels.items():
        count = (series == score).sum()
        rows_q11b.append({"behavior": behavior, "score": score, "score_label": label, "pct": round(count/total*100, 1), "n": total})

q11b_df = pd.DataFrame(rows_q11b)

print("\nN respondants per behavior (excl. NA, excl. Other):")
print(q11b_df.groupby("behavior")["n"].first().to_string())
print()
display(q11b_df.pivot(index="behavior", columns="score_label", values="pct"))

# N per behavior per gender
print("\nN per behavior per gender:")
for col, behavior in q11b_labels.items():
    n_m = pd.to_numeric(df_q11b.loc[df_q11b["gender"]=="Man",   col], errors="coerce").notna().sum()
    n_w = pd.to_numeric(df_q11b.loc[df_q11b["gender"]=="Woman", col], errors="coerce").notna().sum()
    print(f"  {behavior}: Man={n_m}, Woman={n_w}")

# By gender proportions
rows_q11bg = []
for gender in ["Man", "Woman"]:
    subset = df_q11b[df_q11b["gender"] == gender]
    for col, behavior in q11b_labels.items():
        series = pd.to_numeric(subset[col], errors="coerce").dropna()
        total = len(series)
        for score, label in score_labels.items():
            count = (series == score).sum()
            rows_q11bg.append({"gender": gender, "behavior": behavior, "score": score, "score_label": label, "pct": round(count/total*100, 1), "n": total})

q11b_gender_df = pd.DataFrame(rows_q11bg)

pos_q11b_gender = (
    q11b_gender_df[q11b_gender_df["score"].isin([4, 5])]
    .groupby(["gender", "behavior"])["pct"].sum().round(1)
    .reset_index().rename(columns={"pct": "pct_positive"})
)
print("\n% positive (scores 4+5) by behavior and gender:")
display(pos_q11b_gender.pivot(index="behavior", columns="gender", values="pct_positive"))

In [ ]:
# 100% stacked bar — overall
fig = go.Figure()
for score in [1, 2, 3, 4, 5]:
    label = score_labels[score]
    subset = q11b_df[q11b_df["score"] == score]
    fig.add_trace(go.Bar(
        name=label, x=subset["behavior"], y=subset["pct"],
        text=subset["pct"].apply(lambda v: f"{v:.0f}%" if v >= 5 else ""),
        textposition="inside", insidetextanchor="middle",
        marker_color=color_map[label],
    ))
fig.update_layout(
    barmode="stack",
    title="P2_Q11b — Comportements d'evitement la nuit pour raisons de securite",
    xaxis_title="Comportement", yaxis=dict(title="Proportion (%)", range=[0, 100], ticksuffix="%"),
    legend_title="Score", legend=dict(traceorder="normal"), height=520, xaxis_tickangle=-20,
)
fig.show()

# Positive ranking (scores 4+5) — overall
pos_q11b = (
    q11b_df[q11b_df["score"].isin([4, 5])].groupby("behavior")["pct"].sum()
    .reset_index().rename(columns={"pct": "pct_positive"})
    .sort_values("pct_positive", ascending=True)
)
fig2 = px.bar(
    pos_q11b, x="pct_positive", y="behavior", orientation="h",
    text=pos_q11b["pct_positive"].apply(lambda v: f"{v:.1f}%"),
    title="P2_Q11b — Comportements d'evitement les plus frequents (scores 4+5)",
    labels={"pct_positive": "% (score 4 ou 5)", "behavior": "Comportement"},
    color="pct_positive", color_continuous_scale=["#fc8d59", "#ffffbf", "#d73027"], range_color=[0, 100],
)
fig2.update_traces(textposition="outside")
fig2.update_layout(xaxis=dict(range=[0, 108], ticksuffix="%"), coloraxis_showscale=False, height=420)
fig2.show()

# Gender comparison — % positive (4+5) per behavior
fig3 = px.bar(
    pos_q11b_gender,
    x="behavior", y="pct_positive", color="gender",
    barmode="group",
    text=pos_q11b_gender["pct_positive"].astype(str) + "%",
    title="P2_Q11b — Comportements d'evitement la nuit par genre (scores 4+5)",
    labels={"behavior": "Comportement", "pct_positive": "% (score 4 ou 5)", "gender": "Genre"},
    color_discrete_map={"Man": "#4575b4", "Woman": "#d73027"},
    category_orders={"gender": ["Man", "Woman"]},
)
fig3.update_traces(textposition="outside")
fig3.update_layout(
    yaxis=dict(range=[0, 105], ticksuffix="%"),
    xaxis_tickangle=-20, height=500, legend_title="Genre",
)
fig3.show()

# EXPORTS

## Export des lieux de sécurité et insécurité

In [ ]:
for label, df in [("insecurity", PL_spots_insecurity), ("security", PL_spots_security)]:
    df.to_crs(target_crs).to_file(os.path.join(output_file_path_PL + 'WAVE_RYTHM/OUTPUT', f"spots_{label}.gpkg"), driver="GPKG")
    df.to_crs(target_crs).to_parquet(f'{output_file_path_PL}WAVE_RYTHM/OUTPUT/spots_{label}.parquet')
    df.drop(columns="geometry").to_csv(f'{output_file_path_PL}WAVE_RYTHM/OUTPUT/spots_{label}.csv', index=False)
    print(f"Exported spots_{label}: {len(df)} spots")

In [ ]:
# Export zones_girec_map (sous-secteurs GIREC avec comptages secu/insecu)
zones_girec_map.to_crs(target_crs).to_file(os.path.join(output_file_path_PL + 'WAVE_RYTHM/OUTPUT/', "zones_girec_secu_insecu.gpkg"), driver="GPKG")
zones_girec_map.drop(columns="geometry").to_parquet(os.path.join(output_file_path_PL + 'WAVE_RYTHM/OUTPUT/', "zones_girec_secu_insecu.parquet"))
zones_girec_map.drop(columns="geometry").to_csv(os.path.join(output_file_path_PL + 'WAVE_RYTHM/OUTPUT/', "zones_girec_secu_insecu.csv"), index=False)
print(f"Exported zones_girec_map: {len(zones_girec_map)} sous-secteurs")